# B3GK — GPV Multiplicity (K) and Prompting (P)

## Purpose

Continue GPV development from **R4** (B3G Section 4: `dense_recovery` guided masks + D1 dedup + B0 pool; MAE 69.91, S1–S5 12.39, S6 587.6).

B3G Section 6 showed the dense-scene deficit (S4–S6) is **~75–80% merging** (one mask covering many objects) and **~20% missing**. This notebook tackles both:

| Step | Content |
|---|---|
| 1 | Setup |
| 2 | Rebuild R4 and frozen D1 peaks (verified against B3G and the D notebook) |
| **3** | **Merged-mask rule — flagging-quality diagnostic (this section)** |
| 4 | K experiments: K1 (D5-768 peak count), K1b (D1 peak count), K2 / K3 (watershed: shape vs depth gradient), K4 (OmniCount depth carving, their exact parameters), K5 (size-calibrated area ratio) |
| 5 | Comparison; optional single fusion rule; finalise the K rung |
| 6 | P1 (D5-768 prompts in uncovered pixels); freeze GPV |

Naming: **K** = multiplicity options, **P** = prompting, **M1** remains the Marigold experiment.

## Rules

- dev100 only; the held-out cache stays closed until B4.
- GT used for evaluation/diagnosis only, never at inference.
- Rules are written down before each run; every variant is reported.

# 1. Setup

In [33]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
# ============================================================
# Step 1.1 — Paths, repo modules, versions
# ============================================================

import importlib
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"

DEV100_CSV = RESULTS_ROOT / "B1_experiments" / "subsets" / "FSC147_val_dev_100.csv"
B3_REFERENCE_FILE = (RESULTS_ROOT / "B3_B0_B2_M_Geometry_Integration" / "tables"
                     / "B3_step3F_frozen_architectures_per_image.csv")
SG_V11_PER_IMAGE = (RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "section4_SG_v1_1"
                    / "B3R_s4_SG_v1_1_per_image.csv")
B3G_R4_PER_IMAGE = (RESULTS_ROOT / "B3G_Guided_SAM2_GPV" / "rung4_5_6a"
                    / "B3G_rung4_6a_per_image.csv")

B3GK_DIR = RESULTS_ROOT / "B3GK_GPV_Multiplicity_and_Prompts"
B3GK_DIR.mkdir(parents=True, exist_ok=True)

for p in [DEV100_CSV, B3_REFERENCE_FILE, SG_V11_PER_IMAGE, B3G_R4_PER_IMAGE]:
    assert p.exists(), f"Missing: {p}"

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.gpv as gpv
import src.datasets as datasets_mod
for m in (io_cache, gpv, datasets_mod):
    importlib.reload(m)
assert hasattr(gpv, "merged_mask_flags"), "Old gpv.py — commit the B3GK version."
assert hasattr(io_cache, "load_guided_masks"), "Old io_cache.py — commit the B3GK version."

print("Repo commit :", REPO_COMMIT)
print("Output      :", B3GK_DIR)

Repo commit : c1e9172c4044e323e19892370113c5aa34fcb250
Output      : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3GK_GPV_Multiplicity_and_Prompts


In [35]:
# ============================================================
# Step 1.2 — dev100, GT points (evaluation only), references
# ============================================================

dev100_df = pd.read_csv(DEV100_CSV).sort_values("dataset_index").reset_index(drop=True)
assert len(dev100_df) == 100

val_ds = datasets_mod.FSC147Dataset(FSC147_RAW, split="val", load_images=False)
GT_POINTS = {}
for row in dev100_df.itertuples(index=False):
    s = val_ds[row.dataset_index]
    (cls, cat), = s["categories"].items()
    assert s["image_id"] == row.image_id and cat["count"] == row.gt_count
    GT_POINTS[row.dataset_index] = np.asarray(cat["points"], dtype=np.float64).reshape(-1, 2)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]

ref = (pd.read_csv(B3_REFERENCE_FILE)[["dataset_index", "B0_count", "S_count", "SG_count"]]
       .rename(columns={"SG_count": "SG_v1_count"}))
v11 = (pd.read_csv(SG_V11_PER_IMAGE)[["dataset_index", "SG_count"]]
       .rename(columns={"SG_count": "SG_v1_1_count"}))
r4_ref = pd.read_csv(B3G_R4_PER_IMAGE)[["dataset_index", "R4_count"]]
reference_df = (dev100_df.merge(ref, on="dataset_index")
                .merge(v11, on="dataset_index").merge(r4_ref, on="dataset_index"))
assert len(reference_df) == 100
print("References loaded: B0, S, SG_v1, SG_v1.1, R4 (B3G)")

References loaded: B0, S, SG_v1, SG_v1.1, R4 (B3G)


# 2. Rebuild R4 and frozen D1 peaks

- **R4:** `dense_recovery` checkpoint masks → D1 (IoU-NMS 0.5) → pool with frozen B0 masks → IoU-NMS 0.5. Must equal B3G's `R4_count` on 100/100 images.
- **D1 peaks:** frozen configuration from the D notebook (Step 11D.8): per-image min-max normalised SAN target score, `min_distance = round(0.020 × short side)`, `threshold_abs = 0.30`. Per-stratum totals must equal the D notebook (1082, 1233, 1390, 1663, 1143, 1141).

In [36]:
# ============================================================
# Step 2.1 — Build R4 masks and D1 peaks for every dev100 image
# ============================================================

IOU_THRESHOLD = 0.5
BASE_SYSTEM = "dense_recovery"

R4 = {}          # dataset_index -> dict(masks, scores, source)
D1_PEAKS = {}    # dataset_index -> (K, 2) [x, y]
SAN = {}         # dataset_index -> dict(target_score, background_score)

t0 = time.time()
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="R4 + D1"):
    di = row.dataset_index
    g = io_cache.load_guided_masks(io_cache.guided_cache_path(BASE_SYSTEM, di, row.image_id))
    d1_keep = gpv.nms_masks(g["masks"], g["scores"], IOU_THRESHOLD)
    b0_masks, b0_meta = io_cache.load_b0_masks(io_cache.cache_paths(di, row.image_id)["b0_masks"])
    km, ks, src = gpv.build_r4(g["masks"][d1_keep], g["scores"][d1_keep],
                               b0_masks, b0_meta["predicted_iou"], IOU_THRESHOLD)
    R4[di] = {"masks": km, "scores": ks, "source": src}

    san = io_cache.load_san_scores(
        io_cache.SAN_DEV100_CACHE_DIR / io_cache.san_scores_filename(row.image_id))
    SAN[di] = {"target_score": san["target_score"], "background_score": san["background_score"]}
    D1_PEAKS[di] = gpv.d1_peaks(san["target_score"])

print(f"Runtime: {time.time() - t0:.0f} s")

# ---- Checks
r4_counts = np.array([R4[di]["masks"].shape[0] for di in reference_df["dataset_index"]])
r4_match = int((r4_counts == reference_df["R4_count"].to_numpy()).sum())
print(f"R4 count == B3G R4_count : {r4_match}/100  {'PASS' if r4_match == 100 else 'FAIL'}")

d1_totals = (pd.Series({di: len(p) for di, p in D1_PEAKS.items()})
             .rename("d1").to_frame().join(dev100_df.set_index("dataset_index")["density_stratum"])
             .groupby("density_stratum")["d1"].sum().reindex(STRATA))
expected_d1 = pd.Series([1082, 1233, 1390, 1663, 1143, 1141], index=STRATA)
print("\nD1 peaks per stratum (rebuilt vs D notebook):")
display(pd.DataFrame({"rebuilt": d1_totals, "D notebook": expected_d1}))
D1_PASS = bool((d1_totals.values == expected_d1.values).all())
print("D1 reproduction:", "PASS" if D1_PASS else "DIFF — inspect before using D1 peaks")

R4 + D1:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 8 s
R4 count == B3G R4_count : 100/100  PASS

D1 peaks per stratum (rebuilt vs D notebook):


,rebuilt,D notebook
S1_0_20,1082,1082
S2_20_40,1233,1233
S3_40_60,1390,1390
S4_60_80,1663,1663
S5_80_95,1143,1143
S6_95_100,1141,1141


D1 reproduction: PASS


# 3. Merged-mask rule — flagging-quality diagnostic

## Purpose

Every K method acts only on masks flagged as **merged** (covering several objects). The flag must be GT-free. This step measures how well candidate rules detect truly merged masks, **before** any K method is run.

## Candidate rules (peaks = frozen D1 peaks inside the mask)

| Rule | Definition | Concern it addresses |
|---|---|---|
| **A** | area > 4 × median kept area **and** ≥ 2 peaks | Original proposal (6a/6b group definition + peak evidence) |
| **B** | ≥ 2 peaks | Catches flocks where most masks are merged (median contaminated) |
| **C** | ≥ 3 peaks | Fewer false flags than B |
| **D** | area > 4 × median area of **1-peak masks** **and** ≥ 2 peaks (fallback: all-mask median if < 3 one-peak masks) | A's median contamination, while keeping a size check |

## Truth (evaluation only)

A kept mask is **truly merged** if it contains ≥ 2 GT points. Its **merge units** = GT inside − 1 (objects lost if it counts as 1).

## Metrics (overall and per stratum)

- **Precision:** share of flagged masks that are truly merged.
- **Recall (masks):** share of truly merged masks that are flagged.
- **Recall (units):** share of all merge units sitting in flagged masks — the most relevant measure, since a K method can only recover units in flagged masks.
- **False flags in S1–S3:** risk of over-counting sparse scenes.

D1 peaks are known to over-generate in sparse scenes (S1: ~54 peaks per image vs ~10 GT), so rules B/C may flag many single-object masks there — this is what the diagnostic will show.

## Selection

The rule is chosen on **flag quality** (units recall with acceptable sparse false flags), not on count accuracy, and the choice is recorded in the dissertation.

In [5]:
# ============================================================
# Step 3.1 — Per-mask features and truth
# ============================================================

RULES = ["A", "B", "C", "D"]
mask_rows = []

for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="Mask features"):
    di = row.dataset_index
    masks = R4[di]["masks"]
    if masks.shape[0] == 0:
        continue
    peaks = gpv.points_per_mask(masks, D1_PEAKS[di])
    gt_in = gpv.points_per_mask(masks, GT_POINTS[di])
    areas = masks.reshape(masks.shape[0], -1).sum(axis=1)
    med_all = float(np.median(areas))
    flags = {r: gpv.merged_mask_flags(masks, peaks, r) for r in RULES}
    for j in range(masks.shape[0]):
        mask_rows.append({
            "dataset_index": di, "density_stratum": row.density_stratum,
            "source": R4[di]["source"][j],
            "area": int(areas[j]), "area_over_median": areas[j] / med_all,
            "d1_peaks": int(peaks[j]), "gt_inside": int(gt_in[j]),
            "truly_merged": bool(gt_in[j] >= 2),
            "merge_units": int(max(gt_in[j] - 1, 0)),
            **{f"flag_{r}": bool(flags[r][j]) for r in RULES},
        })

masks_df = pd.DataFrame(mask_rows)
print(f"Kept masks: {len(masks_df)} | truly merged: {int(masks_df['truly_merged'].sum())} "
      f"| merge units: {int(masks_df['merge_units'].sum())}")

print("\nD1 peaks inside masks, by truth (median / mean):")
display(masks_df.groupby(["density_stratum", "truly_merged"])["d1_peaks"]
        .agg(["count", "median", "mean"]).round(1))

Mask features:   0%|          | 0/100 [00:00<?, ?it/s]

Kept masks: 3886 | truly merged: 282 | merge units: 6614

D1 peaks inside masks, by truth (median / mean):


count  median  mean
density_stratum truly_merged                     
S1_0_20         False           273     2.0   3.6
                True              7    22.0  29.0
S2_20_40        False           444     2.0   2.6
                True              9     1.0  16.6
S3_40_60        False           536     1.0   2.1
                True             40     3.0   9.2
S4_60_80        False           830     1.0   1.5
                True             55     2.0  11.8
S5_80_95        False           829     1.0   1.1
                True             47     1.0   8.2
S6_95_100       False           692     1.0   0.9
                True            124     1.0   5.6

In [6]:
# ============================================================
# Step 3.2 — Flagging quality per rule (overall and per stratum)
# ============================================================

def flag_quality(df, rule):
    f = df[f"flag_{rule}"]
    t = df["truly_merged"]
    tp = (f & t).sum()
    return {
        "flagged": int(f.sum()),
        "precision": float(tp / f.sum()) if f.sum() else np.nan,
        "recall_masks": float(tp / t.sum()) if t.sum() else np.nan,
        "recall_units": float(df.loc[f, "merge_units"].sum() / df["merge_units"].sum())
                        if df["merge_units"].sum() else np.nan,
        "false_flags": int((f & ~t).sum()),
        "false_flags_gt1": int((f & (df["gt_inside"] == 1)).sum()),
        "false_flags_gt0": int((f & (df["gt_inside"] == 0)).sum()),
    }

overall = pd.DataFrame({r: flag_quality(masks_df, r) for r in RULES}).T
print("Overall flagging quality:")
display(overall.round(3))

rows = []
for st in STRATA:
    sub = masks_df[masks_df["density_stratum"] == st]
    for r in RULES:
        rows.append({"stratum": st, "rule": r, **flag_quality(sub, r)})
per_stratum = pd.DataFrame(rows)

for metric in ["recall_units", "precision", "false_flags"]:
    print(f"\n{metric} by stratum:")
    display(per_stratum.pivot(index="stratum", columns="rule", values=metric)
            .reindex(STRATA).round(3))

sparse = masks_df[masks_df["density_stratum"].isin(STRATA[:3])]
print("\nFalse flags in S1–S3 (sparse-scene risk):")
display(pd.DataFrame({r: flag_quality(sparse, r) for r in RULES}).T[
    ["flagged", "false_flags", "false_flags_gt1", "false_flags_gt0"]])

Overall flagging quality:


,flagged,precision,recall_masks,recall_units,false_flags,false_flags_gt1,false_flags_gt0
A,150.0,0.340,0.181,0.927,99.0,44.0,55.0
B,1271.0,0.086,0.387,0.964,1162.0,1002.0,160.0
C,733.0,0.094,0.245,0.886,664.0,571.0,93.0
D,325.0,0.163,0.188,0.928,272.0,198.0,74.0



recall_units by stratum:


rule,A,B,C,D
stratum,,,,
S1_0_20,0.921,0.974,0.974,0.921
S2_20_40,0.769,0.808,0.769,0.769
S3_40_60,0.636,0.826,0.788,0.644
S4_60_80,0.925,0.961,0.914,0.926
S5_80_95,0.598,0.935,0.923,0.598
S6_95_100,0.969,0.972,0.881,0.969



precision by stratum:


rule,A,B,C,D
stratum,,,,
S1_0_20,0.417,0.036,0.044,0.208
S2_20_40,0.143,0.017,0.018,0.031
S3_40_60,0.412,0.093,0.113,0.095
S4_60_80,0.450,0.122,0.126,0.322
S5_80_95,0.148,0.072,0.145,0.143
S6_95_100,0.424,0.276,0.343,0.424



false_flags by stratum:


rule,A,B,C,D
stratum,,,,
S1_0_20,7,160,131,19
S2_20_40,18,229,165,94
S3_40_60,10,264,181,76
S4_60_80,22,260,111,40
S5_80_95,23,194,53,24
S6_95_100,19,55,23,19



False flags in S1–S3 (sparse-scene risk):


,flagged,false_flags,false_flags_gt1,false_flags_gt0
A,50.0,35.0,15.0,20.0
B,690.0,653.0,569.0,84.0
C,509.0,477.0,428.0,49.0
D,205.0,189.0,158.0,31.0


In [7]:
# ============================================================
# Step 3.3 — Largest-deficit images: are their merged masks flagged?
# ============================================================

big = (masks_df[masks_df["truly_merged"]]
       .groupby("dataset_index")
       .agg(merge_units=("merge_units", "sum"))
       .reset_index())
for r in RULES:
    big[f"units_flagged_{r}"] = [
        int(masks_df[(masks_df["dataset_index"] == di) & masks_df[f"flag_{r}"]]["merge_units"].sum())
        for di in big["dataset_index"]]
big = big.merge(dev100_df[["dataset_index", "image_id", "category", "density_stratum", "gt_count"]],
                on="dataset_index").sort_values("merge_units", ascending=False)
print("Top 12 images by merge units — units inside flagged masks, per rule:")
display(big.head(12)[["image_id", "category", "density_stratum", "gt_count", "merge_units"]
                     + [f"units_flagged_{r}" for r in RULES]])

Top 12 images by merge units — units inside flagged masks, per rule:


,image_id,category,density_stratum,gt_count,merge_units,units_flagged_A,units_flagged_B,units_flagged_C,units_flagged_D
17,935.jpg,birds,S6_95_100,2092,2076,2075,2075,2075,2075
57,6969.jpg,birds,S6_95_100,949,966,935,935,935,935
34,3665.jpg,toilet paper rolls,S6_95_100,907,886,886,886,886,886
19,975.jpg,birds,S6_95_100,718,484,453,453,4,453
29,3426.jpg,polka dots,S6_95_100,219,434,432,432,432,432
60,7436.jpg,pills,S6_95_100,283,263,263,263,263,263
44,5740.jpg,polka dots,S5_80_95,87,171,0,171,171,0
21,1935.jpg,bottle caps,S5_80_95,175,127,115,115,115,115
56,6931.jpg,birds,S6_95_100,355,92,40,40,40,40
47,6061.jpg,seagulls,S5_80_95,104,85,76,79,77,76


In [8]:
# ============================================================
# Step 3.4 — Save
# ============================================================

S3_DIR = B3GK_DIR / "step3_merged_mask_rule"
S3_DIR.mkdir(parents=True, exist_ok=True)
masks_df.to_csv(S3_DIR / "B3GK_s3_mask_features.csv", index=False)
overall.to_csv(S3_DIR / "B3GK_s3_flag_quality_overall.csv")
per_stratum.to_csv(S3_DIR / "B3GK_s3_flag_quality_by_stratum.csv", index=False)
big.to_csv(S3_DIR / "B3GK_s3_top_deficit_images.csv", index=False)

with open(S3_DIR / "B3GK_s3_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3GK_GPV_Multiplicity_and_Prompts",
        "step": "3 — merged-mask rule flagging diagnostic",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT,
        "base": "R4 (dense_recovery + D1 + B0 pool, IoU-NMS 0.5)",
        "peaks": "frozen D1 (min_distance 0.020 x short side, threshold 0.30 on min-max SAN target)",
        "rules": {"A": "area > 4x median(all) AND peaks >= 2",
                  "B": "peaks >= 2", "C": "peaks >= 3",
                  "D": "area > 4x median(1-peak masks; fallback all) AND peaks >= 2"},
        "truth": "truly merged = >= 2 GT points inside (evaluation only)",
        "r4_reproduction": f"{r4_match}/100",
        "d1_reproduction": D1_PASS,
        "heldout_accessed": False, "test_set_accessed": False,
    }, f, indent=2)
print("Saved to:", S3_DIR)

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3GK_GPV_Multiplicity_and_Prompts/step3_merged_mask_rule


## Step 3 decision — merged-mask rule

**Rule A adopted:** a kept R4 mask is *merged* if its area > 4 × the image's median kept-mask area **and** it contains ≥ 2 frozen D1 peaks.

Chosen on **flag quality** (not count accuracy): units recall 0.927, precision 0.34, 150 flagged masks, 35 false flags in S1–S3. Rules B/C flag ~1,000 single-object masks (D1 over-generates on single objects); rule D matches A's recall with ~3× the false flags.

**Known miss:** images where nearly every mask is merged, so the median is itself a merged size (5740.jpg, polka dots, 171 units).

**Nested masks:** large masks can contain objects already segmented individually (e.g. 7436.jpg). K methods therefore act on the **region** (merged mask minus non-merged kept masks), never the whole mask.

# 4a. K experiments on CPU — K0, K1b, K2, K3, K4, K5

## Common protocol (pre-declared, identical for every K)

- **Flagged** = rule A.
- **A_ref** = median area of the image's non-flagged kept masks (fallback: all kept masks) — the single-object size reference.
- **Regions:** flagged masks in descending score order; each region = flagged mask − pixels of non-flagged kept masks − regions already assigned.
- A region **qualifies** if its area ≥ 0.5 × A_ref; otherwise it contributes **0** (the flagged mask only duplicates objects already counted).
- **Count** = number of non-flagged kept masks + Σ region multiplicities; each qualifying region contributes **≥ 1**.

## Estimators

| K | Multiplicity of a qualifying region | Prior |
|---|---|---|
| **K0** | 1 | — (isolates the effect of the region rule itself) |
| **K1b** | number of frozen D1 peaks in the region | Semantic (SAN, 640) |
| **K2** | marker watershed on the distance transform; markers = h-maxima (h = 1 px); segments ≥ 0.25 × A_ref | Shape (non-depth control) |
| **K3** | marker watershed on the depth-gradient magnitude (Sobel of Gaussian-smoothed depth, σ = 1); markers = h-minima, h = 0.1 × (p95 − p5) of the region's gradient; segments ≥ 0.25 × A_ref | Geometric (depth edges) |
| **K4** | OmniCount depth carving with their released parameters (tolerance 0.2, ±5 px window, opening radius 5); number of connected components | Geometric (competitor mechanism) |
| **K5** | round(region area / A_ref) | Size |

K1 (D5-768 peaks) follows in Section 4b (GPU).

**Inputs:** frozen dev100 caches (checkpoint guided masks, B0 masks, SAN maps, Marigold depth). No new inference.

## What to look for

- **K0 vs R4:** does the region rule alone (dropping pure container masks) change the count?
- **S4–S6** gains vs **S1–S3** losses for each K.
- **K2 vs K3:** shape vs depth edges under identical watershed rules.
- **K4** in S6: does OmniCount's opening delete small objects, as predicted?
- **Region calibration:** predicted multiplicity vs GT points inside regions.

In [9]:
# ============================================================
# Step 4a.1 — Run K0, K1b, K2, K3, K4, K5 on every dev100 image
# ============================================================

import src.pipeline.multiplicity as K
importlib.reload(K)

K_NAMES = ["K0", "K1b", "K2", "K3", "K4", "K5"]
img_rows, region_rows = [], []
t0 = time.time()

for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="K experiments"):
    di = row.dataset_index
    masks, scores = R4[di]["masks"], R4[di]["scores"]
    out = {"dataset_index": di, "image_id": row.image_id, "category": row.category,
           "density_stratum": row.density_stratum, "gt_count": row.gt_count,
           "R4_count": int(masks.shape[0])}
    if masks.shape[0] == 0:
        for k in K_NAMES:
            out[f"{k}_count"] = 0
        out.update({"flagged": 0, "qualifying_regions": 0, "a_ref": np.nan})
        img_rows.append(out)
        continue

    peaks = gpv.points_per_mask(masks, D1_PEAKS[di])
    flagged = gpv.merged_mask_flags(masks, peaks, "A")
    a_ref = K.reference_area(masks, flagged)
    depth = io_cache.load_depth(io_cache.cache_paths(di, row.image_id)["depth"])
    min_seg = 0.25 * a_ref

    estimators = {
        "K0": lambda R: 1,
        "K1b": lambda R: K.k1_peaks(R, D1_PEAKS[di]),
        "K2": lambda R: K.k2_watershed_shape(R, min_seg),
        "K3": lambda R: K.k3_watershed_depth_gradient(R, depth, min_seg),
        "K4": lambda R: K.k4_omnicount_carving(R, depth),
        "K5": lambda R: K.k5_area_ratio(R, a_ref),
    }

    regions = K.build_regions(masks, scores, flagged)
    gt_xy = GT_POINTS[di]
    n_nonflag = int((~flagged).sum())
    totals = {k: n_nonflag for k in K_NAMES}
    n_qual = 0
    for mi, region in regions:
        area = int(region.sum())
        qualifies = area > 0 and area >= 0.5 * a_ref
        gt_in = int(gpv.points_per_mask(region[None], gt_xy)[0]) if area else 0
        rr = {"dataset_index": di, "density_stratum": row.density_stratum,
              "mask_index": mi, "region_area": area, "a_ref": a_ref,
              "qualifies": qualifies, "gt_in_region": gt_in}
        for k, f in estimators.items():
            m = max(1, int(f(region))) if qualifies else 0
            totals[k] += m
            rr[k] = m
        n_qual += int(qualifies)
        region_rows.append(rr)

    for k in K_NAMES:
        out[f"{k}_count"] = totals[k]
    out.update({"flagged": int(flagged.sum()), "qualifying_regions": n_qual, "a_ref": a_ref})
    img_rows.append(out)

k_df = pd.DataFrame(img_rows).sort_values("dataset_index").reset_index(drop=True)
k_regions_df = pd.DataFrame(region_rows)
print(f"Runtime: {time.time() - t0:.0f} s | flagged masks: {int(k_df['flagged'].sum())} "
      f"| qualifying regions: {int(k_df['qualifying_regions'].sum())}")

K experiments:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 75 s | flagged masks: 150 | qualifying regions: 133


In [10]:
# ============================================================
# Step 4a.2 — Metrics vs references
# ============================================================

def count_metrics(pred, gt):
    err = np.asarray(pred, float) - np.asarray(gt, float)
    gt = np.asarray(gt, float)
    return {"MAE": float(np.abs(err).mean()), "RMSE": float(np.sqrt((err ** 2).mean())),
            "NAE": float((np.abs(err) / gt).mean()), "mean_signed_error": float(err.mean())}


def system_summary(name, pred, frame):
    gt = frame["gt_count"].to_numpy()
    pred = np.asarray(pred, float)
    s15 = (frame["density_stratum"] != "S6_95_100").to_numpy()
    out = {"system": name, **count_metrics(pred, gt),
           "MAE_S1_S5": float(np.abs(pred[s15] - gt[s15]).mean()),
           "MAE_S6": float(np.abs(pred[~s15] - gt[~s15]).mean())}
    for st in STRATA:
        m = (frame["density_stratum"] == st).to_numpy()
        out[f"MAE_{st[:2]}"] = float(np.abs(pred[m] - gt[m]).mean())
    return out


assert (k_df["dataset_index"].to_numpy() == reference_df["dataset_index"].to_numpy()).all()
assert (k_df["R4_count"].to_numpy() == reference_df["R4_count"].to_numpy()).all()

systems = [("B0", reference_df["B0_count"]), ("S", reference_df["S_count"]),
           ("SG_v1", reference_df["SG_v1_count"]), ("SG_v1.1", reference_df["SG_v1_1_count"]),
           ("R4 (GPV base)", reference_df["R4_count"])]
systems += [(f"R4 + {k}", k_df[f"{k}_count"]) for k in K_NAMES]
k_summary_df = pd.DataFrame([system_summary(n, v, reference_df) for n, v in systems])

COLS = ["system", "MAE", "RMSE", "NAE", "mean_signed_error", "MAE_S1_S5", "MAE_S6",
        "MAE_S1", "MAE_S2", "MAE_S3", "MAE_S4", "MAE_S5"]
display(k_summary_df[COLS].round(2))

,system,MAE,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,B0,80.69,270.23,0.56,-77.47,20.44,622.9,5.15,6.55,14.95,34.30,62.1
1,S,72.00,262.24,0.43,-65.32,14.57,588.9,4.55,5.60,8.95,24.15,44.6
2,SG_v1,61.93,246.59,0.44,-53.27,14.26,491.0,5.95,6.80,8.70,22.40,40.6
3,SG_v1.1,61.99,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5
4,R4 (GPV base),69.91,263.05,0.42,-61.59,12.39,587.6,4.65,7.05,9.35,16.95,35.5
5,R4 + K0,70.06,263.17,0.42,-61.76,12.50,588.1,4.60,7.05,9.40,17.45,35.5
6,R4 + K1b,67.92,250.48,0.54,-50.44,14.06,552.7,8.15,10.40,9.40,22.00,26.6
7,R4 + K2,75.10,260.38,0.68,-48.56,19.24,577.8,8.80,13.00,17.80,22.25,49.5
8,R4 + K3,67.18,163.18,1.21,-3.30,33.17,373.3,19.05,24.90,28.15,28.95,96.4
9,R4 + K4,74.19,262.20,0.52,-54.51,15.16,605.5,5.95,10.45,11.60,18.20,44.0


In [11]:
# ============================================================
# Step 4a.3 — Region calibration and totals by stratum
# ============================================================

q = k_regions_df[k_regions_df["qualifies"]]
print(f"Qualifying regions: {len(q)} | GT points inside them: {int(q['gt_in_region'].sum())}")

calib = pd.DataFrame({
    k: {"predicted_units": int(q[k].sum()),
        "GT_inside": int(q["gt_in_region"].sum()),
        "MAE_per_region": float((q[k] - q["gt_in_region"]).abs().mean()),
        "signed_per_region": float((q[k] - q["gt_in_region"]).mean())}
    for k in K_NAMES}).T
print("\nRegion-level calibration (all strata):")
display(calib.round(2))

calib_st = (q.groupby("density_stratum")[K_NAMES + ["gt_in_region"]].sum()
            .reindex(STRATA))
print("\nPredicted units in qualifying regions vs GT inside, by stratum:")
display(calib_st)

nonq = k_regions_df[~k_regions_df["qualifies"]]
print(f"\nNon-qualifying (container) regions: {len(nonq)} | GT inside them: "
      f"{int(nonq['gt_in_region'].sum())}")

totals_st = (k_df.groupby("density_stratum")[["R4_count"] + [f"{k}_count" for k in K_NAMES]
                                             + ["gt_count"]].sum().reindex(STRATA))
print("\nTotal predicted count vs GT, by stratum:")
display(totals_st)

Qualifying regions: 133 | GT points inside them: 5027

Region-level calibration (all strata):


,predicted_units,GT_inside,MAE_per_region,signed_per_region
K0,133.0,5027.0,37.71,-36.80
K1b,1265.0,5027.0,37.88,-28.29
K2,1453.0,5027.0,43.32,-26.87
K3,5979.0,5027.0,40.42,7.16
K4,858.0,5027.0,41.15,-31.35
K5,16621.0,5027.0,119.37,87.17



Predicted units in qualifying regions vs GT inside, by stratum:


,K0,K1b,K2,K3,K4,K5,gt_in_region
density_stratum,,,,,,,
S1_0_20,11,82,95,300,38,300,2
S2_20_40,21,88,140,378,89,2040,14
S3_40_60,16,58,258,447,68,3446,18
S4_60_80,30,491,336,486,103,4080,314
S5_80_95,27,160,375,886,166,1789,95
S6_95_100,28,386,249,3482,394,4966,4584



Non-qualifying (container) regions: 17 | GT inside them: 0

Total predicted count vs GT, by stratum:


,R4_count,K0_count,K1b_count,K2_count,K3_count,K4_count,K5_count,gt_count
density_stratum,,,,,,,,
S1_0_20,280,279,350,363,568,306,568,195
S2_20_40,453,453,520,572,810,521,2472,320
S3_40_60,576,575,617,817,1006,627,4005,537
S4_60_80,885,875,1336,1181,1331,948,4925,1096
S5_80_95,876,876,1009,1224,1735,1015,2638,1211
S6_95_100,816,811,1169,1032,4265,1177,5749,6686


In [12]:
# ============================================================
# Step 4a.4 — Largest-deficit images and save
# ============================================================

top = k_df.sort_values("gt_count", ascending=False).head(15)
print("15 largest-GT images:")
display(top[["image_id", "category", "density_stratum", "gt_count", "R4_count"]
            + [f"{k}_count" for k in K_NAMES] + ["flagged", "qualifying_regions"]])

S4_DIR = B3GK_DIR / "step4a_K_cpu"
S4_DIR.mkdir(parents=True, exist_ok=True)
k_df.to_csv(S4_DIR / "B3GK_s4a_per_image.csv", index=False)
k_regions_df.to_csv(S4_DIR / "B3GK_s4a_regions.csv", index=False)
k_summary_df.to_csv(S4_DIR / "B3GK_s4a_metrics.csv", index=False)
calib.to_csv(S4_DIR / "B3GK_s4a_region_calibration.csv")
totals_st.to_csv(S4_DIR / "B3GK_s4a_totals_by_stratum.csv")

with open(S4_DIR / "B3GK_s4a_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3GK_GPV_Multiplicity_and_Prompts",
        "step": "4a — K0, K1b, K2, K3, K4, K5 (CPU)",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT,
        "base": "R4", "merged_mask_rule": "A",
        "protocol": {"A_ref": "median area of non-flagged kept masks (fallback all)",
                     "regions": "descending score; minus non-flagged masks and earlier regions",
                     "qualify": "region area >= 0.5 * A_ref, else 0",
                     "min_contribution": 1},
        "estimators": {"K0": "1", "K1b": "D1 peaks in region",
                       "K2": "watershed on distance transform; h-maxima h=1; segments >= 0.25 A_ref",
                       "K3": "watershed on depth-gradient (Sobel, Gaussian sigma 1); h-minima h=0.1*(p95-p5); segments >= 0.25 A_ref",
                       "K4": "OmniCount carving: tol 0.2, +/-5 window, opening radius 5; components",
                       "K5": "round(region area / A_ref)"},
        "inputs": "frozen dev100 caches", "new_inference": False,
        "heldout_accessed": False, "test_set_accessed": False, "gt_used_for_inference": False,
    }, f, indent=2)
print("Saved to:", S4_DIR)

15 largest-GT images:


,image_id,category,density_stratum,gt_count,R4_count,K0_count,K1b_count,K2_count,K3_count,K4_count,K5_count,flagged,qualifying_regions
29,935.jpg,birds,S6_95_100,2092,12,11,110,28,1411,11,1014,3,2
93,6969.jpg,birds,S6_95_100,949,26,26,104,71,592,95,1500,2,2
56,3665.jpg,toilet paper rolls,S6_95_100,907,14,14,53,14,110,14,123,1,1
32,975.jpg,birds,S6_95_100,718,71,71,75,74,1303,74,1325,2,2
25,840.jpg,books,S6_95_100,637,56,56,66,60,73,56,77,3,3
92,6931.jpg,birds,S6_95_100,355,97,97,101,166,107,107,883,1,1
97,7436.jpg,pills,S6_95_100,283,286,286,288,345,354,556,342,1,1
77,5866.jpg,grapes,S6_95_100,267,117,117,150,129,159,128,194,9,9
0,215.jpg,grapes,S6_95_100,259,120,120,153,128,130,123,191,6,6
46,3426.jpg,polka dots,S6_95_100,219,17,13,69,17,26,13,100,5,1


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3GK_GPV_Multiplicity_and_Prompts/step4a_K_cpu


## Step 4a summary

| | MAE | RMSE | S1–S5 | S6 | S5 |
|---|---|---|---|---|---|
| SG_v1.1 | 61.99 | 246.6 | 14.32 | 491.0 | 41.5 |
| R4 | 69.91 | 263.1 | 12.39 | 587.6 | 35.5 |
| R4 + K1b (D1 peaks) | 67.92 | 250.5 | 14.06 | 552.7 | **26.6** |
| R4 + K2 (shape watershed) | 75.10 | 260.4 | 19.24 | 577.8 | 49.5 |
| **R4 + K3 (depth-gradient watershed)** | 67.18 | **163.2** | 33.17 | **373.3** | 96.4 |
| R4 + K4 (OmniCount carving) | 74.19 | 262.2 | 15.16 | 605.5 | 44.0 |
| R4 + K5 (area ratio) | 162.14 | 430.9 | 130.97 | 442.7 | 170.7 |

- **Depth edges (K3) are the strongest multiplicity signal in dense scenes** (S6 373 vs shape K2 578; flocks: 935.jpg 12 → 1,411 of 2,092).
- **Every estimator amplifies rule-A false flags in sparse scenes** (S1–S3 regions hold 34 GT; K3 adds 1,125 units). The problem is *where* K acts, not *how* it counts.
- K4 (OmniCount's mechanism) is worse than R4 in S6, as predicted (opening removes small objects). K5 fails (regions include background).

**Next (pre-declared):** 4b — K1 with CLIP-Surgery D5-768 peaks; 4c — parameter-free gating/consensus: (1) apply K only where SG_v1.1's frozen two-stage gate fires; (2) region multiplicity = median(K1b, K2, K3); plus min(K1b, K3) as the parameter-free cap. A tuned cap (K3 ≤ c × peaks) is **not** used: c would be chosen after seeing dev100 results.

# 4b. K1 — CLIP-Surgery (D5-768) peak counting (GPU)

## Pre-declared

- **D5-768** exactly as frozen in the B1 D notebook (Step 11H): CLIP Surgery `xmed-lab/CLIP_Surgery` @ `d4696d4`, CS-ViT-B/16, input resized to 768 × 768 (bicubic), prompt-ensemble text features, redundant feature from `""`, `get_similarity_map` (already min-max normalised; not renormalised).
- **Peaks:** min_distance = round(0.020 × short side), threshold 0.50 (D notebook selection).
- **K1:** multiplicity of a qualifying region = number of D5-768 peaks inside it; same protocol as 4a (rule A, A_ref, regions, qualify ≥ 0.5 A_ref, ≥ 1).
- D5 maps and peaks are **cached** for dev100 (P1 will reuse them).

## Checks

Per-stratum D5 peak totals vs the D notebook (2000 / 2545 / 2441 / 2962 / 2112 / 2065). Small differences are possible (current CUDA environment); they are reported, not tuned.

## Run order

**T4 runtime** → Sections 1, 2, 4a → 4b. (Section 3 optional.)

In [13]:
# ============================================================
# Step 4b.1 — Install CLIP Surgery (frozen commit) and load CS-ViT-B/16
# ============================================================

import torch
assert torch.cuda.is_available(), "Switch the runtime to T4."
DEVICE = "cuda"

import src.pipeline.semantic as semantic
importlib.reload(semantic); importlib.reload(gpv)

CS_REPO = Path("/content/CLIP_Surgery")
CS_COMMIT = semantic.CLIP_SURGERY_COMMIT
if not CS_REPO.exists():
    !git clone -q https://github.com/xmed-lab/CLIP_Surgery.git /content/CLIP_Surgery
!git -C /content/CLIP_Surgery checkout -q {CS_COMMIT}
!pip install -q ftfy regex

cs_commit = subprocess.check_output(["git", "-C", str(CS_REPO), "rev-parse", "HEAD"], text=True).strip()
assert cs_commit == CS_COMMIT, cs_commit

clip_cs, cs_model = semantic.build_clip_surgery(CS_REPO, DEVICE)
print("CLIP Surgery commit:", cs_commit)
print("clip module        :", clip_cs.__file__)
print("GPU                :", torch.cuda.get_device_name(0), "| torch", torch.__version__,
      "| CUDA", torch.version.cuda)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00


/content/CLIP_Surgery/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
100%|███████████████████████████████████████| 335M/335M [00:03<00:00, 94.8MiB/s]


CLIP Surgery commit: d4696d47f49cfe70f49140afe5eb94f94c5f59bc
clip module        : /content/CLIP_Surgery/clip/__init__.py
GPU                : Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0


In [14]:
# ============================================================
# Step 4b.2 — D5-768 maps + peaks for dev100 (cached)
# ============================================================

from PIL import Image

FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
D5_CACHE = PROJECT_ROOT / "cache" / "D5_768" / "FSC147_val_dev_100"
D5_CACHE.mkdir(parents=True, exist_ok=True)

D5_PEAKS = {}
t0 = time.time()
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="D5-768"):
    di = row.dataset_index
    path = D5_CACHE / f"{int(di):05d}_{io_cache.image_stem(row.image_id)}_D5_768.npz"
    if path.exists():
        with np.load(path) as d:
            sim = d["similarity"].astype(np.float32)
    else:
        image = Image.open(FSC147_IMAGE_DIR / row.image_id).convert("RGB")
        sim = semantic.run_clip_surgery(clip_cs, cs_model, image, row.category, DEVICE)
        np.savez_compressed(path, similarity=sim.astype(np.float32),
                            category=np.array(row.category), dataset_index=np.int32(di))
    D5_PEAKS[di] = gpv.extract_peaks(sim, semantic.D5_MIN_DISTANCE_FRAC,
                                     semantic.D5_SCORE_THRESHOLD)
print(f"Runtime: {time.time() - t0:.0f} s | cache: {D5_CACHE}")

d5_totals = (pd.Series({di: len(p) for di, p in D5_PEAKS.items()}).rename("d5").to_frame()
             .join(dev100_df.set_index("dataset_index")["density_stratum"])
             .groupby("density_stratum")["d5"].sum().reindex(STRATA))
expected_d5 = pd.Series([2000, 2545, 2441, 2962, 2112, 2065], index=STRATA)
cmp_d5 = pd.DataFrame({"rebuilt": d5_totals, "D notebook": expected_d5,
                       "diff": d5_totals - expected_d5})
display(cmp_d5)
D5_EXACT = bool((cmp_d5["diff"] == 0).all())
print("D5-768 reproduction:", "EXACT" if D5_EXACT else
      f"DIFFERS by {int(cmp_d5['diff'].abs().sum())} peaks in total — reported, not tuned")

D5-768:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 37 s | cache: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/cache/D5_768/FSC147_val_dev_100


,rebuilt,D notebook,diff
S1_0_20,2010,2000,10
S2_20_40,2550,2545,5
S3_40_60,2474,2441,33
S4_60_80,2968,2962,6
S5_80_95,2105,2112,-7
S6_95_100,2030,2065,-35


D5-768 reproduction: DIFFERS by 96 peaks in total — reported, not tuned


In [15]:
# ============================================================
# Step 4b.3 — K1 (D5-768 peaks) with the 4a protocol
# ============================================================

k1_rows, k1_region_rows = [], []
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="K1"):
    di = row.dataset_index
    masks, scores = R4[di]["masks"], R4[di]["scores"]
    if masks.shape[0] == 0:
        k1_rows.append({"dataset_index": di, "K1_count": 0}); continue
    peaks = gpv.points_per_mask(masks, D1_PEAKS[di])
    flagged = gpv.merged_mask_flags(masks, peaks, "A")
    a_ref = K.reference_area(masks, flagged)
    total = int((~flagged).sum())
    for mi, region in K.build_regions(masks, scores, flagged):
        area = int(region.sum())
        qualifies = area > 0 and area >= 0.5 * a_ref
        m = max(1, K.k1_peaks(region, D5_PEAKS[di])) if qualifies else 0
        total += m
        k1_region_rows.append({"dataset_index": di, "mask_index": mi, "K1": m})
    k1_rows.append({"dataset_index": di, "K1_count": total})

k1_df = pd.DataFrame(k1_rows)
k_df = k_df.drop(columns=["K1_count"], errors="ignore").merge(k1_df, on="dataset_index")
k_regions_df = k_regions_df.drop(columns=["K1"], errors="ignore").merge(
    pd.DataFrame(k1_region_rows), on=["dataset_index", "mask_index"], how="left")
k_regions_df["K1"] = k_regions_df["K1"].fillna(0).astype(int)

systems_b = [("SG_v1.1", reference_df["SG_v1_1_count"]), ("R4 (GPV base)", reference_df["R4_count"])]
systems_b += [(f"R4 + {k}", k_df[f"{k}_count"]) for k in ["K1", "K1b", "K2", "K3", "K4", "K5"]]
k_summary_b = pd.DataFrame([system_summary(n, v, reference_df) for n, v in systems_b])
display(k_summary_b[COLS].round(2))

K1:   0%|          | 0/100 [00:00<?, ?it/s]

,system,MAE,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,SG_v1.1,61.99,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5
1,R4 (GPV base),69.91,263.05,0.42,-61.59,12.39,587.6,4.65,7.05,9.35,16.95,35.5
2,R4 + K1,69.42,229.05,0.79,-37.66,21.33,502.2,10.10,21.10,15.90,27.50,42.8
3,R4 + K1b,67.92,250.48,0.54,-50.44,14.06,552.7,8.15,10.40,9.40,22.00,26.6
4,R4 + K2,75.10,260.38,0.68,-48.56,19.24,577.8,8.80,13.00,17.80,22.25,49.5
5,R4 + K3,67.18,163.18,1.21,-3.30,33.17,373.3,19.05,24.90,28.15,28.95,96.4
6,R4 + K4,74.19,262.20,0.52,-54.51,15.16,605.5,5.95,10.45,11.60,18.20,44.0
7,R4 + K5,162.14,430.87,4.25,103.12,130.97,442.7,19.05,108.00,176.80,200.15,170.7


In [16]:
# ============================================================
# Step 4b.4 — Region calibration incl. K1, and K3/peak ratios
# ============================================================

q = k_regions_df[k_regions_df["qualifies"]]
calib_b = (q.groupby("density_stratum")[["K1", "K1b", "K2", "K3", "gt_in_region"]]
           .sum().reindex(STRATA))
calib_b["K3/K1"] = calib_b["K3"] / calib_b["K1"].replace(0, np.nan)
calib_b["K3/K1b"] = calib_b["K3"] / calib_b["K1b"].replace(0, np.nan)
print("Predicted units in qualifying regions vs GT inside, by stratum:")
display(calib_b.round(2))
print("If K3/K1 is LOWER in S6 than in S1–S3, a peak-based cap could separate the "
      "regimes (to be decided before any cap is tested); if HIGHER, a cap cannot work.")

S4B_DIR = B3GK_DIR / "step4b_K1_D5"
S4B_DIR.mkdir(parents=True, exist_ok=True)
k_df.to_csv(S4B_DIR / "B3GK_s4b_per_image_with_K1.csv", index=False)
k_regions_df.to_csv(S4B_DIR / "B3GK_s4b_regions_with_K1.csv", index=False)
k_summary_b.to_csv(S4B_DIR / "B3GK_s4b_metrics.csv", index=False)
calib_b.to_csv(S4B_DIR / "B3GK_s4b_region_calibration.csv")
cmp_d5.to_csv(S4B_DIR / "B3GK_s4b_d5_reproduction.csv")
with open(S4B_DIR / "B3GK_s4b_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3GK_GPV_Multiplicity_and_Prompts", "step": "4b — K1 (D5-768 peaks)",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT, "clip_surgery_commit": cs_commit,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "cuda": torch.version.cuda,
        "d5": {"resolution": 768, "min_distance_frac": semantic.D5_MIN_DISTANCE_FRAC,
               "threshold": semantic.D5_SCORE_THRESHOLD, "renormalised": False},
        "d5_reproduction_exact": D5_EXACT, "d5_cache": str(D5_CACHE),
        "heldout_accessed": False, "test_set_accessed": False,
    }, f, indent=2)
print("Saved to:", S4B_DIR)

Predicted units in qualifying regions vs GT inside, by stratum:


,K1,K1b,K2,K3,gt_in_region,K3/K1,K3/K1b
density_stratum,,,,,,,
S1_0_20,121,82,95,300,2,2.48,3.66
S2_20_40,302,88,140,378,14,1.25,4.30
S3_40_60,222,58,258,447,18,2.01,7.71
S4_60_80,635,491,336,486,314,0.77,0.99
S5_80_95,372,160,375,886,95,2.38,5.54
S6_95_100,891,386,249,3482,4584,3.91,9.02


If K3/K1 is LOWER in S6 than in S1–S3, a peak-based cap could separate the regimes (to be decided before any cap is tested); if HIGHER, a cap cannot work.
Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3GK_GPV_Multiplicity_and_Prompts/step4b_K1_D5


## Step 4b summary

- **D5-768 reproduction:** 96 of ~14,100 peaks differ (0.7%; S6 −1.7%) — CUDA 13.0 vs the original environment; reported, not tuned.
- **K1 (D5-768 peaks):** S6 502.2 (better than K1b 552.7, close to SG_v1.1 491.0) but S1–S5 21.33; in qualifying S6 regions K1 predicts 891 units vs 4,584 GT (K3: 3,482). Semantic peaks, even at 768, cannot separate S6's tiny packed objects.
- **K3 / peak ratios are highest in S6** (K3/K1 3.91; K3/K1b 9.02) — a peak-based cap cannot separate the regimes. The tuned-cap option stays excluded.
- **Division of labour:** S1–S4 none (R4); S5 K1b (26.6); S6 K3 (373.3).

# 4c. Gating and consensus (CPU, no new inference)

## Pre-declared variants

Declared in the Step-4a summary (before 4b ran), plus one extra added after 4b:

| Variant | Rule |
|---|---|
| **G(K)** — SG gate | Apply estimator K only in images where **SG_v1.1's frozen two-stage gate fires** (`stage2_on`); elsewhere keep the R4 count. Reported for K1b, K2, K3 and K1. |
| **Med3** | Region multiplicity = **median(K1b, K2, K3)** |
| **Min2** | Region multiplicity = **min(K1b, K3)** (parameter-free cap; expected to fail — K3/peak ratios are highest in S6) |
| **Med3-K1** *(added after 4b)* | Region multiplicity = **median(K1, K2, K3)** |
| **G(Med3)** | Med3, applied only where the SG gate fires |

All variants keep the 4a protocol (rule A, A_ref, regions, qualifying ≥ 0.5 A_ref, ≥ 1 per qualifying region). No parameter is introduced.

## Inputs

Saved 4b tables (per image and per region) and SG_v1.1's per-image gate states (B3R Section 4). Only **Section 1** needs to run first; CPU is enough.

In [17]:
# ============================================================
# Step 4c.1 — Load saved 4a/4b tables and SG_v1.1 gate states
# ============================================================

# Metric helpers (defined in 4a.2; redefined here so 4c can run after Section 1 only)
if "system_summary" not in globals():
    def count_metrics(pred, gt):
        err = np.asarray(pred, float) - np.asarray(gt, float)
        gt = np.asarray(gt, float)
        return {"MAE": float(np.abs(err).mean()), "RMSE": float(np.sqrt((err ** 2).mean())),
                "NAE": float((np.abs(err) / gt).mean()), "mean_signed_error": float(err.mean())}

    def system_summary(name, pred, frame):
        gt = frame["gt_count"].to_numpy()
        pred = np.asarray(pred, float)
        s15 = (frame["density_stratum"] != "S6_95_100").to_numpy()
        out = {"system": name, **count_metrics(pred, gt),
               "MAE_S1_S5": float(np.abs(pred[s15] - gt[s15]).mean()),
               "MAE_S6": float(np.abs(pred[~s15] - gt[~s15]).mean())}
        for st in STRATA:
            m = (frame["density_stratum"] == st).to_numpy()
            out[f"MAE_{st[:2]}"] = float(np.abs(pred[m] - gt[m]).mean())
        return out

COLS = ["system", "MAE", "RMSE", "NAE", "mean_signed_error", "MAE_S1_S5", "MAE_S6",
        "MAE_S1", "MAE_S2", "MAE_S3", "MAE_S4", "MAE_S5"]

S4B_DIR = B3GK_DIR / "step4b_K1_D5"
k_img = pd.read_csv(S4B_DIR / "B3GK_s4b_per_image_with_K1.csv").sort_values("dataset_index").reset_index(drop=True)
k_reg = pd.read_csv(S4B_DIR / "B3GK_s4b_regions_with_K1.csv")

sg_gate = pd.read_csv(SG_V11_PER_IMAGE)[["dataset_index", "stage1_on", "stage2_on"]]
k_img = k_img.merge(sg_gate, on="dataset_index", validate="one_to_one")
assert (k_img["dataset_index"].to_numpy() == reference_df["dataset_index"].to_numpy()).all()

# Non-flagged base per image = K0 count minus its qualifying-region contributions
k0_reg = k_reg.groupby("dataset_index")["K0"].sum()
k_img["nonflag_base"] = k_img["K0_count"] - k_img["dataset_index"].map(k0_reg).fillna(0).astype(int)

print(f"Images: {len(k_img)} | regions: {len(k_reg)} "
      f"(qualifying {int(k_reg['qualifies'].sum())})")
print(f"SG_v1.1 gate fires in {int(k_img['stage2_on'].sum())} images:")
display(k_img[k_img["stage2_on"]].groupby("density_stratum").size().reindex(STRATA).fillna(0).astype(int)
        .rename("gate_on_images").to_frame().T)

Images: 100 | regions: 150 (qualifying 133)
SG_v1.1 gate fires in 16 images:


density_stratum,S1_0_20,S2_20_40,S3_40_60,S4_60_80,S5_80_95,S6_95_100
gate_on_images,3,2,3,3,1,4


In [18]:
# ============================================================
# Step 4c.2 — Build every variant's per-image count
# ============================================================

q_mask = k_reg["qualifies"].astype(bool)

def region_rule_count(values):
    # values: per-region multiplicity Series aligned with k_reg (0 for non-qualifying)
    v = pd.Series(np.where(q_mask, values, 0), index=k_reg.index)
    per_img = v.groupby(k_reg["dataset_index"]).sum()
    return (k_img["nonflag_base"] + k_img["dataset_index"].map(per_img).fillna(0)).to_numpy()

def at_least_one(x):
    return np.maximum(1, np.asarray(x))

variants = {}
# Single estimators (from 4a/4b) for reference
for k in ["K1", "K1b", "K2", "K3"]:
    variants[f"{k}"] = k_img[f"{k}_count"].to_numpy()

# Consensus rules
med3 = at_least_one(np.median(k_reg[["K1b", "K2", "K3"]].to_numpy(), axis=1))
med3_k1 = at_least_one(np.median(k_reg[["K1", "K2", "K3"]].to_numpy(), axis=1))
min2 = at_least_one(np.minimum(k_reg["K1b"], k_reg["K3"]))
variants["Med3"] = region_rule_count(med3)
variants["Min2"] = region_rule_count(min2)
variants["Med3-K1 (added after 4b)"] = region_rule_count(med3_k1)

# SG-gated versions: estimator where gate fires, R4 elsewhere
gate = k_img["stage2_on"].to_numpy(bool)
r4 = k_img["R4_count"].to_numpy()
for k in ["K1b", "K2", "K3", "K1"]:
    variants[f"G({k})"] = np.where(gate, k_img[f"{k}_count"], r4)
variants["G(Med3)"] = np.where(gate, variants["Med3"], r4)

rows = [system_summary("SG_v1.1", reference_df["SG_v1_1_count"], reference_df),
        system_summary("R4 (GPV base)", reference_df["R4_count"], reference_df)]
rows += [system_summary(f"R4 + {n}", v, reference_df) for n, v in variants.items()]
summary_4c = pd.DataFrame(rows)
display(summary_4c[COLS].round(2))

,system,MAE,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,SG_v1.1,61.99,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5
1,R4 (GPV base),69.91,263.05,0.42,-61.59,12.39,587.6,4.65,7.05,9.35,16.95,35.5
2,R4 + K1,69.42,229.05,0.79,-37.66,21.33,502.2,10.10,21.10,15.90,27.50,42.8
3,R4 + K1b,67.92,250.48,0.54,-50.44,14.06,552.7,8.15,10.40,9.40,22.00,26.6
4,R4 + K2,75.10,260.38,0.68,-48.56,19.24,577.8,8.80,13.00,17.80,22.25,49.5
5,R4 + K3,67.18,163.18,1.21,-3.30,33.17,373.3,19.05,24.90,28.15,28.95,96.4
6,R4 + Med3,70.81,249.18,0.63,-49.21,16.36,560.9,9.45,12.25,13.00,21.00,35.8
7,R4 + Min2,68.24,250.77,0.51,-54.58,13.54,560.5,8.15,8.30,9.10,20.10,30.6
8,R4 + Med3-K1 (added after 4b),68.26,227.03,0.69,-42.00,18.91,512.4,9.30,15.10,15.90,24.50,40.6
9,R4 + G(K1b),69.05,260.18,0.48,-58.99,12.76,575.7,7.45,7.45,9.20,15.55,35.5


In [19]:
# ============================================================
# Step 4c.3 — Where do the variants differ? (gate-on vs gate-off images)
# ============================================================

def mae_subset(pred, mask):
    gt = reference_df["gt_count"].to_numpy()
    return float(np.abs(np.asarray(pred, float)[mask] - gt[mask]).mean()) if mask.any() else np.nan

sparse = reference_df["density_stratum"].isin(STRATA[:3]).to_numpy()
dense = ~sparse
cells = {"gate ON (all)": gate, "gate ON, S1–S3": gate & sparse, "gate ON, S4–S6": gate & dense,
         "gate OFF, S4–S6": ~gate & dense}
tab = pd.DataFrame({name: {c: mae_subset(v, m) for c, m in cells.items()}
                    for name, v in [("R4", r4), ("SG_v1.1", reference_df["SG_v1_1_count"].to_numpy())]
                                    + list(variants.items())}).T
tab.columns = [f"MAE {c} (n={int(m.sum())})" for c, m in cells.items()]
print("MAE in image subsets defined by the SG gate:")
display(tab.round(2))

top = reference_df[["image_id", "category", "density_stratum", "gt_count", "R4_count"]].copy()
for n in ["K3", "G(K3)", "Med3", "Med3-K1 (added after 4b)", "G(Med3)"]:
    top[n] = variants[n]
top["gate"] = gate
print("\n15 largest-GT images:")
display(top.sort_values("gt_count", ascending=False).head(15))

MAE in image subsets defined by the SG gate:


,MAE gate ON (all) (n=16),"MAE gate ON, S1–S3 (n=8)","MAE gate ON, S4–S6 (n=8)","MAE gate OFF, S4–S6 (n=32)"
R4,132.56,9.38,255.75,141.38
SG_v1.1,69.94,13.50,126.38,148.94
K1,114.06,21.00,207.12,135.72
K1b,127.19,17.00,237.38,135.44
K2,129.94,21.50,238.38,150.34
K3,115.25,57.88,172.62,121.72
Med3,128.69,22.12,235.25,140.78
Min2,129.06,17.00,241.12,137.00
Med3-K1 (added after 4b),114.31,21.75,206.88,136.41
G(K1b),127.19,17.00,237.38,141.38



15 largest-GT images:


,image_id,category,density_stratum,gt_count,R4_count,K3,G(K3),Med3,Med3-K1 (added after 4b),G(Med3),gate
29,935.jpg,birds,S6_95_100,2092,12,1411,12,120.0,313.0,12.0,False
93,6969.jpg,birds,S6_95_100,949,26,592,592,135.0,209.0,135.0,True
56,3665.jpg,toilet paper rolls,S6_95_100,907,14,110,14,53.0,108.0,14.0,False
32,975.jpg,birds,S6_95_100,718,71,1303,1303,77.0,173.0,77.0,True
25,840.jpg,books,S6_95_100,637,56,73,56,66.0,71.0,56.0,False
92,6931.jpg,birds,S6_95_100,355,97,107,107,107.0,166.0,107.0,True
97,7436.jpg,pills,S6_95_100,283,286,354,286,345.0,345.0,286.0,False
77,5866.jpg,grapes,S6_95_100,267,117,159,117,140.0,143.0,117.0,False
0,215.jpg,grapes,S6_95_100,259,120,130,130,132.0,132.0,132.0,True
46,3426.jpg,polka dots,S6_95_100,219,17,26,17,26.0,26.0,17.0,False


In [20]:
# ============================================================
# Step 4c.4 — Save
# ============================================================

S4C_DIR = B3GK_DIR / "step4c_gating_consensus"
S4C_DIR.mkdir(parents=True, exist_ok=True)
out = k_img[["dataset_index", "image_id", "density_stratum", "gt_count", "R4_count", "stage2_on"]].copy()
for n, v in variants.items():
    out[n] = v
out.to_csv(S4C_DIR / "B3GK_s4c_per_image.csv", index=False)
summary_4c.to_csv(S4C_DIR / "B3GK_s4c_metrics.csv", index=False)
tab.to_csv(S4C_DIR / "B3GK_s4c_gate_subsets.csv")
with open(S4C_DIR / "B3GK_s4c_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3GK_GPV_Multiplicity_and_Prompts", "step": "4c — gating and consensus",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT,
        "pre_declared": {"G(K)": "estimator K where SG_v1.1 stage2_on, else R4",
                         "Med3": "median(K1b, K2, K3) per region",
                         "Min2": "min(K1b, K3) per region",
                         "G(Med3)": "Med3 where SG gate fires, else R4"},
        "added_after_4b": {"Med3-K1": "median(K1, K2, K3) per region"},
        "new_parameters": None, "new_inference": False,
        "heldout_accessed": False, "test_set_accessed": False,
    }, f, indent=2)
print("Saved to:", S4C_DIR)

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3GK_GPV_Multiplicity_and_Prompts/step4c_gating_consensus


## Step 4c summary

- No parameter-free consensus or SG-gated variant unlocks K3's S6 gain without sparse damage. Best: G(K1) 66.95, G(K3) 67.14 vs R4 69.91 and SG_v1.1 61.99.
- **The SG gate is mismatched to GPV:** it misses 935.jpg (2,092 birds; K3 1,411 vs R4 12) and fires in 8 sparse images (gate-on S1–S3 MAE: K3 57.9 vs R4 9.4). In the 32 gate-off S4–S6 images, ungated K3 is best (121.7 vs R4 141.4).
- SG's gate detects *B0 missed a large semantic region*; GPV's failure is *its own masks are merged*.
- Med3-K1 vs Med3: K1 helps S6 (560.9 → 512.4) but hurts S1–S5 (16.36 → 18.91).

# 4d. GPV-native gate (one-shot, pre-declared)

## Rationale

K should act only where **most objects in the image sit inside merged regions rather than in individually segmented masks**. This is an image-level property of GPV's own output, so the gate compares merged evidence with individual evidence in the same image.

## Pre-declared gates (written before any 4d computation; parameter-free)

Using the 4a protocol (rule A flags, A_ref, regions, qualifying ≥ 0.5 × A_ref):

| Gate | Rule (gate ON if …) | Evidence |
|---|---|---|
| **Gate 1 — merging dominates (peaks)** | Σ frozen D1 peaks inside qualifying regions **>** number of non-flagged kept masks | Semantic peaks (object-specific) |
| **Gate 2 — merged area dominates** | Σ qualifying-region area **>** area of the union of non-flagged kept masks | Area |

If an image has no qualifying region, both gates are OFF (count = R4).

**Gated variant:** estimator count where the gate is ON, R4 count where it is OFF. Applied to K1b, K2, K3, K1, Med3 and Med3-K1 (all already computed in 4a–4c).

## Commitment

**One shot.** The result is accepted as is. If neither gate helps clearly, K is not added to GPV (GPV stays R4) and K3 moves to SG (roadmap step 6). No further gate design on dev100.

## Run order

CPU. Sections **1 and 2** (rebuild R4 and D1 peaks), then 4d.

In [21]:
# ============================================================
# Step 4d.1 — Compute Gate 1 and Gate 2 for every image
# ============================================================

import src.pipeline.multiplicity as K
importlib.reload(K)
assert "R4" in globals() and "D1_PEAKS" in globals(), "Run Sections 1 and 2 first."

gate_rows = []
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="GPV gates"):
    di = row.dataset_index
    masks, scores = R4[di]["masks"], R4[di]["scores"]
    rec = {"dataset_index": di, "density_stratum": row.density_stratum, "gt_count": row.gt_count}
    if masks.shape[0] == 0:
        gate_rows.append({**rec, "n_nonflag": 0, "peaks_in_regions": 0, "qual_regions": 0,
                          "area_regions": 0, "area_nonflag_union": 0,
                          "gate1": False, "gate2": False})
        continue
    peaks = gpv.points_per_mask(masks, D1_PEAKS[di])
    flagged = gpv.merged_mask_flags(masks, peaks, "A")
    a_ref = K.reference_area(masks, flagged)
    n_nonflag = int((~flagged).sum())
    nonflag_union = masks[~flagged].any(axis=0) if n_nonflag else np.zeros(masks.shape[1:], bool)

    peaks_in, area_q, n_q = 0, 0, 0
    for mi, region in K.build_regions(masks, scores, flagged):
        area = int(region.sum())
        if area > 0 and area >= 0.5 * a_ref:
            n_q += 1
            area_q += area
            peaks_in += K.k1_peaks(region, D1_PEAKS[di])   # raw count, no floor

    gate_rows.append({**rec, "n_nonflag": n_nonflag, "peaks_in_regions": peaks_in,
                      "qual_regions": n_q, "area_regions": area_q,
                      "area_nonflag_union": int(nonflag_union.sum()),
                      "gate1": bool(n_q > 0 and peaks_in > n_nonflag),
                      "gate2": bool(n_q > 0 and area_q > nonflag_union.sum())})

gates_df = pd.DataFrame(gate_rows).sort_values("dataset_index").reset_index(drop=True)
assert (gates_df["dataset_index"].to_numpy() == reference_df["dataset_index"].to_numpy()).all()

sg_on = pd.read_csv(SG_V11_PER_IMAGE).set_index("dataset_index").loc[
    gates_df["dataset_index"], "stage2_on"].to_numpy(bool)
gates_df["sg_gate"] = sg_on

print("Gate ON images by stratum:")
display(gates_df.groupby("density_stratum")[["gate1", "gate2", "sg_gate"]].sum()
        .reindex(STRATA).astype(int))

GPV gates:   0%|          | 0/100 [00:00<?, ?it/s]

Gate ON images by stratum:


,gate1,gate2,sg_gate
density_stratum,,,
S1_0_20,1,2,3
S2_20_40,1,3,2
S3_40_60,0,4,3
S4_60_80,6,4,3
S5_80_95,0,4,1
S6_95_100,4,5,4


In [22]:
# ============================================================
# Step 4d.2 — Gated variants: metrics
# ============================================================

# Metric helpers (defined in 4a.2; redefined here so 4c can run after Section 1 only)
if "system_summary" not in globals():
    def count_metrics(pred, gt):
        err = np.asarray(pred, float) - np.asarray(gt, float)
        gt = np.asarray(gt, float)
        return {"MAE": float(np.abs(err).mean()), "RMSE": float(np.sqrt((err ** 2).mean())),
                "NAE": float((np.abs(err) / gt).mean()), "mean_signed_error": float(err.mean())}

    def system_summary(name, pred, frame):
        gt = frame["gt_count"].to_numpy()
        pred = np.asarray(pred, float)
        s15 = (frame["density_stratum"] != "S6_95_100").to_numpy()
        out = {"system": name, **count_metrics(pred, gt),
               "MAE_S1_S5": float(np.abs(pred[s15] - gt[s15]).mean()),
               "MAE_S6": float(np.abs(pred[~s15] - gt[~s15]).mean())}
        for st in STRATA:
            m = (frame["density_stratum"] == st).to_numpy()
            out[f"MAE_{st[:2]}"] = float(np.abs(pred[m] - gt[m]).mean())
        return out

COLS = ["system", "MAE", "RMSE", "NAE", "mean_signed_error", "MAE_S1_S5", "MAE_S6",
        "MAE_S1", "MAE_S2", "MAE_S3", "MAE_S4", "MAE_S5"]

S4C_DIR = B3GK_DIR / "step4c_gating_consensus"
v = pd.read_csv(S4C_DIR / "B3GK_s4c_per_image.csv").sort_values("dataset_index").reset_index(drop=True)
assert (v["dataset_index"].to_numpy() == gates_df["dataset_index"].to_numpy()).all()

ESTIMATORS = {"K1b": "K1b", "K2": "K2", "K3": "K3", "K1": "K1",
              "Med3": "Med3", "Med3-K1": "Med3-K1 (added after 4b)"}
r4 = v["R4_count"].to_numpy()

rows = [system_summary("SG_v1.1", reference_df["SG_v1_1_count"], reference_df),
        system_summary("R4 (GPV base)", reference_df["R4_count"], reference_df),
        system_summary("K3 (ungated)", v["K3"], reference_df),
        system_summary("G_SG(K3) (4c)", v["G(K3)"], reference_df)]
gated = {}
for g in ["gate1", "gate2"]:
    on = gates_df[g].to_numpy(bool)
    for name, col in ESTIMATORS.items():
        pred = np.where(on, v[col].to_numpy(), r4)
        gated[f"{g}({name})"] = pred
        rows.append(system_summary(f"{g}({name})", pred, reference_df))
summary_4d = pd.DataFrame(rows)
display(summary_4d[COLS].round(2))

,system,MAE,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,SG_v1.1,61.99,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5
1,R4 (GPV base),69.91,263.05,0.42,-61.59,12.39,587.6,4.65,7.05,9.35,16.95,35.5
2,K3 (ungated),67.18,163.18,1.21,-3.30,33.17,373.3,19.05,24.90,28.15,28.95,96.4
3,G_SG(K3) (4c),67.14,248.88,0.75,-38.94,16.51,522.8,16.90,7.55,16.00,16.10,35.5
4,gate1(K1b),69.27,251.35,0.51,-54.31,14.64,560.9,7.45,8.85,9.35,22.50,35.5
5,gate1(K2),70.42,260.37,0.49,-58.78,13.63,581.5,7.40,7.55,9.35,19.30,35.5
6,gate1(K3),54.32,151.10,0.72,-34.78,18.07,380.6,16.90,7.35,9.35,29.95,35.5
7,gate1(K1),67.40,232.69,0.54,-49.98,15.90,530.9,7.70,9.55,9.35,27.20,35.5
8,gate1(Med3),69.13,249.68,0.52,-55.93,14.47,561.1,8.25,7.55,9.35,22.20,35.5
9,gate1(Med3-K1),66.48,230.52,0.51,-52.16,15.10,528.9,7.70,7.55,9.35,25.60,35.5


In [23]:
# ============================================================
# Step 4d.3 — Gate behaviour: subsets and largest-GT images
# ============================================================

gt = reference_df["gt_count"].to_numpy()
sparse = reference_df["density_stratum"].isin(STRATA[:3]).to_numpy()

def mae_on(pred, m):
    return float(np.abs(np.asarray(pred, float)[m] - gt[m]).mean()) if m.any() else np.nan

for g in ["gate1", "gate2"]:
    on = gates_df[g].to_numpy(bool)
    subsets = {"ON, S1–S3": on & sparse, "ON, S4–S6": on & ~sparse,
               "OFF, S4–S6": ~on & ~sparse}
    t = pd.DataFrame({n: {k: mae_on(p, m) for k, m in subsets.items()}
                      for n, p in [("R4", r4), ("SG_v1.1", reference_df["SG_v1_1_count"].to_numpy()),
                                   ("K3 ungated", v["K3"].to_numpy()),
                                   (f"{g}(K3)", gated[f"{g}(K3)"]),
                                   (f"{g}(K1b)", gated[f"{g}(K1b)"])]}).T
    t.columns = [f"MAE {k} (n={int(m.sum())})" for k, m in subsets.items()]
    print(f"\n=== {g}: MAE by subset ===")
    display(t.round(2))

top = reference_df[["image_id", "category", "density_stratum", "gt_count", "R4_count"]].copy()
top["K3"] = v["K3"].to_numpy()
top["gate1(K3)"] = gated["gate1(K3)"]
top["gate2(K3)"] = gated["gate2(K3)"]
for c in ["peaks_in_regions", "n_nonflag", "gate1", "gate2", "sg_gate"]:
    top[c] = gates_df[c].to_numpy()
print("\n15 largest-GT images:")
display(top.sort_values("gt_count", ascending=False).head(15))

print("\nSparse (S1–S3) images where Gate 1 is ON:")
display(top[sparse & gates_df["gate1"].to_numpy(bool)])


=== gate1: MAE by subset ===


,"MAE ON, S1–S3 (n=2)","MAE ON, S4–S6 (n=10)","MAE OFF, S4–S6 (n=30)"
R4,9.5,425.7,77.10
SG_v1.1,11.0,408.1,56.53
K3 ungated,135.0,244.7,94.30
gate1(K3),135.0,244.7,77.10
gate1(K1b),55.5,410.1,77.10



=== gate2: MAE by subset ===


,"MAE ON, S1–S3 (n=9)","MAE ON, S4–S6 (n=13)","MAE OFF, S4–S6 (n=27)"
R4,6.89,382.31,59.26
SG_v1.1,5.11,324.08,57.93
K3 ungated,110.67,281.23,60.00
gate2(K3),110.67,281.23,59.26
gate2(K1b),14.33,370.62,59.26



15 largest-GT images:


,image_id,category,density_stratum,gt_count,R4_count,K3,gate1(K3),gate2(K3),peaks_in_regions,n_nonflag,gate1,gate2,sg_gate
29,935.jpg,birds,S6_95_100,2092,12,1411,1411,1411,101,9,True,True,False
93,6969.jpg,birds,S6_95_100,949,26,592,592,592,79,24,True,True,True
56,3665.jpg,toilet paper rolls,S6_95_100,907,14,110,110,110,40,13,True,True,False
32,975.jpg,birds,S6_95_100,718,71,1303,71,1303,6,69,False,True,True
25,840.jpg,books,S6_95_100,637,56,73,56,56,13,53,False,False,False
92,6931.jpg,birds,S6_95_100,355,97,107,97,97,5,96,False,False,True
97,7436.jpg,pills,S6_95_100,283,286,354,286,286,3,285,False,False,False
77,5866.jpg,grapes,S6_95_100,267,117,159,117,117,42,108,False,False,False
0,215.jpg,grapes,S6_95_100,259,120,130,120,120,39,114,False,False,True
46,3426.jpg,polka dots,S6_95_100,219,17,26,26,26,57,12,True,True,False



Sparse (S1–S3) images where Gate 1 is ON:


,image_id,category,density_stratum,gt_count,R4_count,K3,gate1(K3),gate2(K3),peaks_in_regions,n_nonflag,gate1,gate2,sg_gate
13,737.jpg,ants,S1_0_20,9,20,265,265,265,65,11,True,True,True
74,5667.jpg,donuts tray,S2_20_40,20,28,34,34,28,37,27,True,False,False


In [24]:
# ============================================================
# Step 4d.4 — Save
# ============================================================

S4D_DIR = B3GK_DIR / "step4d_gpv_gate"
S4D_DIR.mkdir(parents=True, exist_ok=True)
out = gates_df.copy()
for n, p in gated.items():
    out[n] = p
out.to_csv(S4D_DIR / "B3GK_s4d_per_image.csv", index=False)
summary_4d.to_csv(S4D_DIR / "B3GK_s4d_metrics.csv", index=False)
with open(S4D_DIR / "B3GK_s4d_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3GK_GPV_Multiplicity_and_Prompts", "step": "4d — GPV-native gate (one shot)",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT,
        "gates": {"gate1": "sum D1 peaks in qualifying regions > number of non-flagged kept masks",
                  "gate2": "sum qualifying-region area > union area of non-flagged kept masks",
                  "no_qualifying_region": "gate OFF"},
        "gated_variant": "estimator count where gate ON, R4 count where OFF",
        "commitment": "one shot; no further gate design on dev100",
        "new_parameters": None, "new_inference": False,
        "heldout_accessed": False, "test_set_accessed": False,
    }, f, indent=2)
print("Saved to:", S4D_DIR)

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3GK_GPV_Multiplicity_and_Prompts/step4d_gpv_gate


## Step 4d decision — GPV-native gate

**Gate 1 (pre-declared, one shot) adopted.** Σ D1 peaks in qualifying merged regions > number of non-flagged kept masks → apply K3; otherwise keep R4.

| | MAE | RMSE | NAE | S1–S5 | S6 |
|---|---|---|---|---|---|
| SG_v1.1 | 61.99 | 246.6 | 0.44 | 14.32 | 491.0 |
| R4 | 69.91 | 263.1 | 0.42 | 12.39 | 587.6 |
| **gate1(K3)** | **54.32** | **151.1** | 0.72 | 18.07 | 380.6 |

- Gate 1 fires on 935 / 6969 / 3665 / 3426 (merging dominates) and stays off on 975 and 7436 (mostly individually segmented); only 2 sparse false alarms (SG gate: 8). Gate 2 (area) fails (9 sparse alarms).
- **Caveats:** gain concentrated (935.jpg ≈ 14 MAE, 6969.jpg ≈ 5.7); one costly false alarm (737.jpg ants: 20 → 265); S4 worsens (16.95 → 29.95). Not tuned further (one-shot commitment).
- Stratum-reweighted full-set estimate: gate1(K3) ~37, SG_v1.1 ~40, R4 ~42.

**Two GPV versions carried forward:** **GPV-K** = R4 + gate1(K3); **GPV-R4** = R4.

# 5. P1 — D5-768 prompts in uncovered pixels (GPU)

## Purpose

Address the ~20% of the dense-scene deficit that is **missing** objects (B3G Section 6): objects inside no kept mask.

## Pre-declared rules

1. **Prompts:** frozen D5-768 peaks (Step 4b cache; min_distance 0.020 × short side, threshold 0.50) lying on pixels **not covered by any kept R4 mask** (union of all kept masks).
2. **Masks:** SAM2.1-S point prompt, single positive point, `multimask_output=False` (checkpoint protocol; Rung 3 showed multimask selection changes nothing).
3. **Keep a new mask only if** it contains its prompt, is non-empty, and its area ≤ **4 × median kept R4 area** (the existing group-mask convention — a new mask larger than that is a background/group mask, not a missing object).
4. **Pool** = R4 kept masks + accepted new masks → IoU-NMS 0.5 by predicted IoU.
5. **GPV-R4 + P1:** count = kept masks.
6. **GPV-K + P1:** apply the frozen GPV-K steps **after** P1 on the new pool: rule A flags (D1 peaks) → A_ref → regions → Gate 1 → K3 where the gate fires.

No new parameters: every threshold reuses an existing convention or frozen setting.

## Inputs and environment

R4 (Section 2), D1 peaks (Section 2), D5-768 maps (cached, Step 4b), Marigold depth (frozen dev100 cache). New SAM2 masks are generated in the current environment (T4, CUDA 13.0); B3G showed point-prompted SAM2 is bit-identical across environments.

## Run order

**T4 runtime** → Sections 1, 2 → Section 5. (4a–4d not needed; their results are saved.)

In [25]:
# ============================================================
# Step 5.1 — Install SAM2 (frozen commit) and build the predictor
# ============================================================

import torch
assert torch.cuda.is_available(), "Switch the runtime to T4."
DEVICE = "cuda"

import src.pipeline.b0 as b0
import src.pipeline.semantic as semantic
import src.pipeline.multiplicity as K
for m in (b0, semantic, K, gpv):
    importlib.reload(m)

SAM2_REPO = Path("/content/sam2_repo")
FROZEN_SAM2_COMMIT = b0.FROZEN_SAM2_COMMIT
SAM2_CHECKPOINT = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_small.pt"

if not (SAM2_REPO / "sam2" / "__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content

sam2_commit = subprocess.check_output(["git", "-C", str(SAM2_REPO), "rev-parse", "HEAD"], text=True).strip()
assert sam2_commit == FROZEN_SAM2_COMMIT, sam2_commit
if str(SAM2_REPO) not in sys.path:
    sys.path.insert(0, str(SAM2_REPO))
importlib.invalidate_caches()

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

assert b0.sha256_file(SAM2_CHECKPOINT) == b0.FROZEN_CHECKPOINT_SHA256
sam2_model = build_sam2(b0.FROZEN_MODEL_CFG, str(SAM2_CHECKPOINT), device=DEVICE)
sam2_model.eval()
predictor = SAM2ImagePredictor(sam2_model)
print("SAM2 commit:", sam2_commit, "| GPU:", torch.cuda.get_device_name(0),
      "| torch", torch.__version__, "| CUDA", torch.version.cuda)

/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.9 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
/content
SAM2 commit: 2b90b9f5ceec907a1c18123530e92e794ad901a4 | GPU: Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0


In [26]:
# ============================================================
# Step 5.2 — Load D5-768 peaks from the Step-4b cache
# ============================================================

assert "R4" in globals() and "D1_PEAKS" in globals(), "Run Sections 1 and 2 first."
D5_CACHE = PROJECT_ROOT / "cache" / "D5_768" / "FSC147_val_dev_100"

D5_PEAKS = {}
for row in dev100_df.itertuples(index=False):
    path = D5_CACHE / f"{int(row.dataset_index):05d}_{io_cache.image_stem(row.image_id)}_D5_768.npz"
    assert path.exists(), f"Missing D5 cache: {path} (run Step 4b once)"
    with np.load(path) as d:
        sim = d["similarity"].astype(np.float32)
    D5_PEAKS[row.dataset_index] = gpv.extract_peaks(sim, semantic.D5_MIN_DISTANCE_FRAC,
                                                    semantic.D5_SCORE_THRESHOLD)
print("D5-768 peaks loaded:", sum(len(p) for p in D5_PEAKS.values()))

D5-768 peaks loaded: 14137


In [27]:
# ============================================================
# Step 5.3 — P1 prompts, new masks, pooling (+ GPV-K on the new pool)
# ============================================================

from PIL import Image
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
GROUP_FACTOR = 4.0


def predict_point(point_xy):
    masks, scores, _ = predictor.predict(
        point_coords=np.asarray([point_xy], dtype=np.float32),
        point_labels=np.asarray([1], dtype=np.int32),
        multimask_output=False)
    return np.asarray(masks[0], dtype=bool), float(np.asarray(scores)[0])


def gpv_k_count(masks, scores, d1_peaks, depth):
    # Frozen GPV-K steps: rule A -> A_ref -> regions -> Gate 1 -> K3
    if masks.shape[0] == 0:
        return 0, False
    peaks = gpv.points_per_mask(masks, d1_peaks)
    flagged = gpv.merged_mask_flags(masks, peaks, "A")
    a_ref = K.reference_area(masks, flagged)
    n_nonflag = int((~flagged).sum())
    regions = [(mi, r) for mi, r in K.build_regions(masks, scores, flagged)
               if r.sum() > 0 and r.sum() >= 0.5 * a_ref]
    peaks_in = sum(K.k1_peaks(r, d1_peaks) for _, r in regions)
    gate_on = len(regions) > 0 and peaks_in > n_nonflag
    if not gate_on:
        return int(masks.shape[0]), False
    min_seg = 0.25 * a_ref
    total = n_nonflag + sum(max(1, K.k3_watershed_depth_gradient(r, depth, min_seg))
                            for _, r in regions)
    return int(total), True


p1_rows = []
t0 = time.time()
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="P1"):
    di = row.dataset_index
    km, ks = R4[di]["masks"], R4[di]["scores"]
    h, w = km.shape[1:] if km.shape[0] else Image.open(FSC147_IMAGE_DIR / row.image_id).size[::-1]
    covered = km.any(axis=0) if km.shape[0] else np.zeros((h, w), bool)
    med_area = float(np.median(km.reshape(km.shape[0], -1).sum(axis=1))) if km.shape[0] else np.inf

    pts = D5_PEAKS[di]
    if len(pts):
        px = np.clip(np.rint(pts[:, 0]).astype(int), 0, w - 1)
        py = np.clip(np.rint(pts[:, 1]).astype(int), 0, h - 1)
        prompts = pts[~covered[py, px]]
    else:
        prompts = np.zeros((0, 2))

    new_m, new_s, rejected_big, rejected_other = [], [], 0, 0
    if len(prompts):
        image_np = np.array(Image.open(FSC147_IMAGE_DIR / row.image_id).convert("RGB"), copy=True)
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts:
                m, s = predict_point(p)
                area = int(m.sum())
                cx = int(np.clip(np.rint(p[0]), 0, w - 1)); cy = int(np.clip(np.rint(p[1]), 0, h - 1))
                if area == 0 or not m[cy, cx]:
                    rejected_other += 1
                elif area > GROUP_FACTOR * med_area:
                    rejected_big += 1
                else:
                    new_m.append(m); new_s.append(s)

    if new_m:
        pool, pool_s, src = gpv.pool_masks([km, np.asarray(new_m)], [ks, np.asarray(new_s)],
                                           ["r4", "p1"])
        keep = gpv.nms_masks(pool, pool_s, IOU_THRESHOLD)
        p1_masks, p1_scores, p1_src = pool[keep], pool_s[keep], src[keep]
    else:
        p1_masks, p1_scores, p1_src = km, ks, np.full(km.shape[0], "r4", dtype=object)

    depth = io_cache.load_depth(io_cache.cache_paths(di, row.image_id)["depth"])
    gpvk_p1, gate_p1 = gpv_k_count(p1_masks, p1_scores, D1_PEAKS[di], depth)

    added = p1_masks[p1_src == "p1"] if p1_masks.shape[0] else p1_masks
    gt_in_added = gpv.points_per_mask(added, GT_POINTS[di]) if added.shape[0] else np.zeros(0, int)

    p1_rows.append({
        "dataset_index": di, "image_id": row.image_id, "density_stratum": row.density_stratum,
        "gt_count": row.gt_count, "R4_count": int(km.shape[0]),
        "d5_peaks": int(len(pts)), "p1_prompts": int(len(prompts)),
        "rejected_big": rejected_big, "rejected_other": rejected_other,
        "accepted_new": len(new_m), "added_after_nms": int((p1_src == "p1").sum()),
        "removed_r4_by_nms": int(km.shape[0] - (p1_src == "r4").sum()),
        "added_gt0": int((gt_in_added == 0).sum()), "added_gt1": int((gt_in_added == 1).sum()),
        "added_gt2plus": int((gt_in_added >= 2).sum()),
        "GPV_R4_P1_count": int(p1_masks.shape[0]),
        "GPV_K_P1_count": gpvk_p1, "gate1_after_P1": gate_p1,
    })

p1_df = pd.DataFrame(p1_rows).sort_values("dataset_index").reset_index(drop=True)
print(f"Runtime: {(time.time() - t0) / 60:.1f} min")

P1:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 1.9 min


In [28]:
# ============================================================
# Step 5.4 — Metrics: GPV-R4, GPV-K, with and without P1
# ============================================================

if "system_summary" not in globals():
    def count_metrics(pred, gt):
        err = np.asarray(pred, float) - np.asarray(gt, float)
        gt = np.asarray(gt, float)
        return {"MAE": float(np.abs(err).mean()), "RMSE": float(np.sqrt((err ** 2).mean())),
                "NAE": float((np.abs(err) / gt).mean()), "mean_signed_error": float(err.mean())}

    def system_summary(name, pred, frame):
        gt = frame["gt_count"].to_numpy()
        pred = np.asarray(pred, float)
        s15 = (frame["density_stratum"] != "S6_95_100").to_numpy()
        out = {"system": name, **count_metrics(pred, gt),
               "MAE_S1_S5": float(np.abs(pred[s15] - gt[s15]).mean()),
               "MAE_S6": float(np.abs(pred[~s15] - gt[~s15]).mean())}
        for st in STRATA:
            m = (frame["density_stratum"] == st).to_numpy()
            out[f"MAE_{st[:2]}"] = float(np.abs(pred[m] - gt[m]).mean())
        return out
COLS = ["system", "MAE", "RMSE", "NAE", "mean_signed_error", "MAE_S1_S5", "MAE_S6",
        "MAE_S1", "MAE_S2", "MAE_S3", "MAE_S4", "MAE_S5"]

assert (p1_df["dataset_index"].to_numpy() == reference_df["dataset_index"].to_numpy()).all()
s4d = pd.read_csv(B3GK_DIR / "step4d_gpv_gate" / "B3GK_s4d_per_image.csv").sort_values("dataset_index")
gpvk = s4d["gate1(K3)"].to_numpy()

# Consistency: where P1 changed nothing, GPV-K + P1 must equal GPV-K
unchanged = ((p1_df["added_after_nms"] == 0) & (p1_df["removed_r4_by_nms"] == 0)).to_numpy()
same = int((p1_df["GPV_K_P1_count"].to_numpy()[unchanged] == gpvk[unchanged]).sum())
print(f"Consistency (images unchanged by P1): GPV-K+P1 == GPV-K on {same}/{int(unchanged.sum())} "
      f"{'PASS' if same == unchanged.sum() else 'CHECK'}")

summary_5 = pd.DataFrame([
    system_summary("SG_v1.1", reference_df["SG_v1_1_count"], reference_df),
    system_summary("GPV-R4", reference_df["R4_count"], reference_df),
    system_summary("GPV-R4 + P1", p1_df["GPV_R4_P1_count"], reference_df),
    system_summary("GPV-K", gpvk, reference_df),
    system_summary("GPV-K + P1", p1_df["GPV_K_P1_count"], reference_df),
])
display(summary_5[COLS].round(2))

print(f"\nGate 1 ON: before P1 {int(s4d['gate1'].sum())} images | after P1 "
      f"{int(p1_df['gate1_after_P1'].sum())} images")

Consistency (images unchanged by P1): GPV-K+P1 == GPV-K on 17/17 PASS


,system,MAE,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,SG_v1.1,61.99,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5
1,GPV-R4,69.91,263.05,0.42,-61.59,12.39,587.6,4.65,7.05,9.35,16.95,35.5
2,GPV-R4 + P1,68.76,258.08,0.64,-49.22,15.60,547.2,9.50,13.80,13.50,22.00,22.8
3,GPV-K,54.32,151.10,0.72,-34.78,18.07,380.6,16.90,7.35,9.35,29.95,35.5
4,GPV-K + P1,52.84,140.58,0.96,-21.96,21.36,336.2,22.20,14.10,13.50,34.90,22.8



Gate 1 ON: before P1 12 images | after P1 12 images


In [29]:
# ============================================================
# Step 5.5 — P1 diagnostics by stratum and save
# ============================================================

diag5 = p1_df.groupby("density_stratum")[[
    "gt_count", "R4_count", "d5_peaks", "p1_prompts", "rejected_big", "rejected_other",
    "accepted_new", "added_after_nms", "removed_r4_by_nms",
    "added_gt0", "added_gt1", "added_gt2plus", "GPV_R4_P1_count"]].sum().reindex(STRATA)
print("P1 diagnostics (totals per stratum; GT columns are evaluation only):")
display(diag5)

S5_DIR = B3GK_DIR / "step5_P1"
S5_DIR.mkdir(parents=True, exist_ok=True)
p1_df.to_csv(S5_DIR / "B3GK_s5_per_image.csv", index=False)
summary_5.to_csv(S5_DIR / "B3GK_s5_metrics.csv", index=False)
diag5.to_csv(S5_DIR / "B3GK_s5_diagnostics_by_stratum.csv")
with open(S5_DIR / "B3GK_s5_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3GK_GPV_Multiplicity_and_Prompts", "step": "5 — P1",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT, "sam2_commit": sam2_commit,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "cuda": torch.version.cuda,
        "rules": {"prompts": "D5-768 peaks on pixels not covered by any kept R4 mask",
                  "mask": "SAM2 single positive point, multimask_output=False",
                  "accept": f"non-empty, contains prompt, area <= {GROUP_FACTOR} x median kept R4 area",
                  "pool": "R4 kept + accepted new; IoU-NMS 0.5 by predicted IoU",
                  "GPV-K+P1": "rule A -> A_ref -> regions -> Gate 1 -> K3 on the P1 pool"},
        "new_parameters": None,
        "heldout_accessed": False, "test_set_accessed": False,
    }, f, indent=2)
print("Saved to:", S5_DIR)

P1 diagnostics (totals per stratum; GT columns are evaluation only):


,gt_count,R4_count,d5_peaks,p1_prompts,rejected_big,rejected_other,accepted_new,added_after_nms,removed_r4_by_nms,added_gt0,added_gt1,added_gt2plus,GPV_R4_P1_count
density_stratum,,,,,,,,,,,,,
S1_0_20,195,280,2010,566,246,72,248,116,11,88,27,1,385
S2_20_40,320,453,2550,592,140,77,375,152,9,121,28,3,596
S3_40_60,537,576,2474,625,171,64,390,184,7,117,56,11,753
S4_60_80,1096,885,2968,598,108,84,406,239,8,148,80,11,1116
S5_80_95,1211,876,2105,423,71,50,302,178,1,72,94,12,1053
S6_95_100,6686,816,2030,671,126,27,518,408,4,57,231,120,1220


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3GK_GPV_Multiplicity_and_Prompts/step5_P1


## Step 5 decision — P1 not adopted

| | dev100 MAE | Stratum-reweighted MAE (est.) | S1 | S5 | S6 |
|---|---|---|---|---|---|
| GPV-R4 | 69.91 | ~42.3 | 4.65 | 35.5 | 587.6 |
| GPV-R4 + P1 | 68.76 | ~42.5 | 9.50 | 22.8 | 547.2 |
| GPV-K | 54.32 | ~37.1 | 16.90 | 35.5 | 380.6 |
| GPV-K + P1 | 52.84 | ~37.2 | 22.20 | 22.8 | 336.2 |

- P1 recovers real objects in dense scenes (S6: 231 of 408 added masks contain one object) but adds mostly false positives in sparse scenes (S1: 88 of 116 contain none): sparse images have large uncovered backgrounds where D5 places false peaks.
- The dev100 gain disappears on the stratum-reweighted estimate (dev100 deliberately oversamples dense scenes).
- Same pattern as every "more masks" lever: helps dense, hurts sparse. **Not adopted (parsimony).** Reported as an ablation.

**Methodological note:** dev100 MAE systematically favours dense-scene levers; the stratum-reweighted MAE (population weights S1–S4 0.20, S5 0.15, S6 0.05) is reported alongside it from here on.

# 6. Freeze GPV — GPV_R4_v1 and GPV_K_v1

Two GPV versions are frozen for B4 (held-out FSC-147 validation):

| Version | Definition |
|---|---|
| **GPV_R4_v1** | `dense_recovery` guided masks → D1 dedup (IoU-NMS 0.5) → pool with B0 AMG masks → IoU-NMS 0.5. Count = kept masks. |
| **GPV_K_v1** | GPV_R4_v1 + rule A merged-mask flags + Gate 1 + K3 depth-gradient watershed multiplicity (`multiplicity.gpv_k_count`). |

This section (a) verifies that the frozen function reproduces the dev100 counts exactly, (b) writes both frozen configs with every rule, parameter and selection decision, and (c) records dev100 reference metrics, including the stratum-reweighted MAE.

**Run order:** CPU → Sections 1, 2 → Section 6. Commit the updated `src/pipeline/multiplicity.py` first.

In [37]:
# ============================================================
# Step 6.1 — Reproduce GPV_R4_v1 and GPV_K_v1 with the frozen function
# ============================================================

import src.pipeline.multiplicity as K
importlib.reload(K)
assert hasattr(K, "gpv_k_count"), "Old multiplicity.py — commit the freeze version."
assert "R4" in globals() and "D1_PEAKS" in globals(), "Run Sections 1 and 2 first."

s4d = pd.read_csv(B3GK_DIR / "step4d_gpv_gate" / "B3GK_s4d_per_image.csv").sort_values("dataset_index")

frozen_rows = []
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="Frozen GPV"):
    di = row.dataset_index
    depth = io_cache.load_depth(io_cache.cache_paths(di, row.image_id)["depth"])
    count_k, gate = K.gpv_k_count(R4[di]["masks"], R4[di]["scores"], D1_PEAKS[di], depth)
    frozen_rows.append({"dataset_index": di, "image_id": row.image_id,
                        "category": row.category, "density_stratum": row.density_stratum,
                        "gt_count": row.gt_count,
                        "GPV_R4_v1_count": int(R4[di]["masks"].shape[0]),
                        "GPV_K_v1_count": count_k, "gate1_on": gate})
frozen_df = pd.DataFrame(frozen_rows).sort_values("dataset_index").reset_index(drop=True)

r4_ok = int((frozen_df["GPV_R4_v1_count"].to_numpy() == reference_df["R4_count"].to_numpy()).sum())
k_ok = int((frozen_df["GPV_K_v1_count"].to_numpy() == s4d["gate1(K3)"].to_numpy()).sum())
g_ok = int((frozen_df["gate1_on"].to_numpy() == s4d["gate1"].to_numpy(bool)).sum())
print(f"GPV_R4_v1 == B3G R4      : {r4_ok}/100  {'PASS' if r4_ok == 100 else 'FAIL'}")
print(f"GPV_K_v1  == Step 4d     : {k_ok}/100  {'PASS' if k_ok == 100 else 'FAIL'}")
print(f"Gate 1    == Step 4d     : {g_ok}/100  {'PASS' if g_ok == 100 else 'FAIL'}")
FREEZE_REPRO_PASS = r4_ok == 100 and k_ok == 100 and g_ok == 100

Frozen GPV:   0%|          | 0/100 [00:00<?, ?it/s]

GPV_R4_v1 == B3G R4      : 100/100  PASS
GPV_K_v1  == Step 4d     : 100/100  PASS
Gate 1    == Step 4d     : 100/100  PASS


In [38]:
# ============================================================
# Step 6.2 — Reference metrics (dev100 + stratum-reweighted)
# ============================================================

STRATUM_WEIGHTS = {"S1_0_20": 0.20, "S2_20_40": 0.20, "S3_40_60": 0.20,
                   "S4_60_80": 0.20, "S5_80_95": 0.15, "S6_95_100": 0.05}

def reweighted_mae(pred):
    gt = reference_df["gt_count"].to_numpy()
    err = np.abs(np.asarray(pred, float) - gt)
    st = reference_df["density_stratum"].to_numpy()
    return float(sum(w * err[st == s].mean() for s, w in STRATUM_WEIGHTS.items()))

if "system_summary" not in globals():
    def count_metrics(pred, gt):
        err = np.asarray(pred, float) - np.asarray(gt, float)
        gt = np.asarray(gt, float)
        return {"MAE": float(np.abs(err).mean()), "RMSE": float(np.sqrt((err ** 2).mean())),
                "NAE": float((np.abs(err) / gt).mean()), "mean_signed_error": float(err.mean())}

    def system_summary(name, pred, frame):
        gt = frame["gt_count"].to_numpy()
        pred = np.asarray(pred, float)
        s15 = (frame["density_stratum"] != "S6_95_100").to_numpy()
        out = {"system": name, **count_metrics(pred, gt),
               "MAE_S1_S5": float(np.abs(pred[s15] - gt[s15]).mean()),
               "MAE_S6": float(np.abs(pred[~s15] - gt[~s15]).mean())}
        for st in STRATA:
            m = (frame["density_stratum"] == st).to_numpy()
            out[f"MAE_{st[:2]}"] = float(np.abs(pred[m] - gt[m]).mean())
        return out

ref_systems = [("B0", reference_df["B0_count"]), ("S", reference_df["S_count"]),
               ("SG_v1", reference_df["SG_v1_count"]), ("SG_v1.1", reference_df["SG_v1_1_count"]),
               ("GPV_R4_v1", frozen_df["GPV_R4_v1_count"]), ("GPV_K_v1", frozen_df["GPV_K_v1_count"])]
freeze_metrics = pd.DataFrame([{**system_summary(n, p, reference_df),
                                "MAE_reweighted": reweighted_mae(p)} for n, p in ref_systems])
display(freeze_metrics[["system", "MAE", "MAE_reweighted", "RMSE", "NAE", "MAE_S1_S5", "MAE_S6",
                        "MAE_S1", "MAE_S2", "MAE_S3", "MAE_S4", "MAE_S5"]].round(2))
print("Stratum-reweighted MAE uses population weights; dev100's S6 tail is heavier than the "
      "population's, so it remains an estimate. B4 measures the real distribution.")

,system,MAE,MAE_reweighted,RMSE,NAE,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,B0,80.69,52.65,270.23,0.56,20.44,622.9,5.15,6.55,14.95,34.30,62.1
1,S,72.00,44.78,262.24,0.43,14.57,588.9,4.55,5.60,8.95,24.15,44.6
2,SG_v1,61.93,39.41,246.59,0.44,14.26,491.0,5.95,6.80,8.70,22.40,40.6
3,SG_v1.1,61.99,39.52,246.59,0.44,14.32,491.0,5.70,6.80,8.60,22.60,41.5
4,GPV_R4_v1,69.91,42.31,263.05,0.42,12.39,587.6,4.65,7.05,9.35,16.95,35.5
5,GPV_K_v1,54.32,37.06,151.10,0.72,18.07,380.6,16.90,7.35,9.35,29.95,35.5


Stratum-reweighted MAE uses population weights; dev100's S6 tail is heavier than the population's, so it remains an estimate. B4 measures the real distribution.


In [39]:
# ============================================================
# Step 6.3 — Write frozen configs and reference counts
# ============================================================

assert FREEZE_REPRO_PASS, "Reproduction failed — do not freeze."

def metrics_for(name):
    r = freeze_metrics.set_index("system").loc[name]
    return {k: round(float(r[k]), 4) for k in
            ["MAE", "MAE_reweighted", "RMSE", "NAE", "MAE_S1_S5", "MAE_S6",
             "MAE_S1", "MAE_S2", "MAE_S3", "MAE_S4", "MAE_S5"]}

common = {
    "status": "FROZEN_FOR_HELDOUT_FSC147_VALIDATION",
    "development_dataset": "FSC-147 validation dev100",
    "prompt_type": "text (class name only); training-free",
    "implementation": {"repo_commit": REPO_COMMIT,
                       "modules": ["src/pipeline/gpv.py", "src/pipeline/multiplicity.py",
                                   "src/pipeline/io_cache.py"]},
    "validation_policy": {"next_stage": "B4 — 1,186 held-out FSC-147 validation images",
                          "parameters_may_be_retuned_before_validation": False,
                          "FSC147_test_set_reserved": True, "test_set_accessed": False},
}

r4_core = {
    "guided_prompts": {"source": "End-to-End checkpoint system 'dense_recovery'",
                       "semantic_points": "F1 fusion: C1 regions + D6B9-strict peaks",
                       "sam2": "SAM2.1 Hiera-Small, single positive point, multimask_output=False"},
    "dedup_D1": {"rule": "IoU-NMS by descending SAM2 predicted IoU", "iou_threshold": 0.5},
    "b0_pool": {"source": "SAM2AutomaticMaskGenerator (default params, apply_postprocessing=False)",
                "merge": "IoU-NMS 0.5 by predicted IoU over guided-D1 U B0 masks"},
    "count": "number of kept masks",
}

gpv_r4 = {"architecture_id": "GPV_R4_v1",
          "architecture_formula": "guided SAM2 (dense_recovery) + D1 dedup + B0 pool",
          **common, "pipeline": r4_core,
          "development_reference_metrics": metrics_for("GPV_R4_v1"),
          "selection_history": [
              "B3G Rung 2: D1 chosen over D2 (D2 worse in all systems); IoU 0.5 by convention (insensitive 0.3-0.7)",
              "B3G: dense_recovery chosen among the 3 existing checkpoint systems on dev100",
              "B3G Rungs 3, 5, 6a, 6b not adopted (no overall gain)",
              "B3GK: K and P1 not added in this version (parsimony; best sparse-scene accuracy)"]}

gpv_k = {"architecture_id": "GPV_K_v1",
         "architecture_formula": "GPV_R4_v1 + rule A + Gate 1 + K3 (depth-gradient watershed)",
         **common,
         "pipeline": {**r4_core,
             "merged_mask_rule_A": {"area": "> 4 x median kept-mask area",
                                    "peaks": ">= 2 frozen D1 peaks inside",
                                    "d1_peaks": "min-max normalised SAN target; min_distance 0.020 x short side; threshold 0.30"},
             "regions": {"A_ref": "median area of non-flagged kept masks (fallback: all)",
                         "order": "flagged masks by descending score",
                         "region": "flagged mask minus non-flagged masks minus earlier regions",
                         "qualify": "area >= 0.5 x A_ref (else contributes 0)"},
             "gate_1": "sum of D1 peaks in qualifying regions > number of non-flagged kept masks",
             "K3": {"depth": "Marigold v1.1 (768, 4 steps, ensemble 1, seed 0, fp16)",
                    "elevation": "Sobel gradient magnitude of Gaussian-smoothed depth (sigma 1)",
                    "markers": "h-minima of gradient in region, h = 0.1 x (p95 - p5)",
                    "segments": ">= 0.25 x A_ref", "min_per_region": 1},
             "count": "gate ON: non-flagged masks + sum of region multiplicities; gate OFF: R4 count",
             "function": "src.pipeline.multiplicity.gpv_k_count"},
         "development_reference_metrics": metrics_for("GPV_K_v1"),
         "known_weaknesses": [
             "gain concentrated: 935.jpg ~14 MAE, 6969.jpg ~5.7 MAE of the improvement over R4",
             "sparse false alarm: 737.jpg (ants) 20 -> 265",
             "S4 worsens (16.95 -> 29.95); NAE rises 0.42 -> 0.72",
             "rule A misses images where nearly every mask is merged (5740.jpg)",
             "G2-style depth noise sensitivity not addressed; K3 over-counts 975.jpg if gated on (it is not)"],
         "selection_history": [
             "Step 3: rule A chosen on flag quality (units recall 0.927, precision 0.34)",
             "Step 4a/4b: K3 strongest dense multiplicity cue (S6 373 vs shape K2 578); K1, K1b, K2, K4, K5 not adopted",
             "Step 4c: SG gate and parameter-free consensus did not help (mismatched gate)",
             "Step 4d: Gate 1 pre-declared, one shot, adopted; Gate 2 rejected; no further gate design",
             "Step 5: P1 not adopted (gain vanishes on stratum-reweighted MAE)",
             "Rejected on principle: tuned cap K3 <= c x peaks (c would be fitted to dev100)"]}

FREEZE_DIR = B3GK_DIR / "frozen"
FREEZE_DIR.mkdir(parents=True, exist_ok=True)
for cfg in (gpv_r4, gpv_k):
    with open(FREEZE_DIR / f"{cfg['architecture_id']}_frozen_config.json", "w") as f:
        json.dump(cfg, f, indent=2)
frozen_df.to_csv(FREEZE_DIR / "GPV_frozen_dev100_reference_counts.csv", index=False)
freeze_metrics.to_csv(FREEZE_DIR / "GPV_frozen_dev100_metrics.csv", index=False)

print("=" * 80)
print("GPV FROZEN")
print("=" * 80)
for cfg in (gpv_r4, gpv_k):
    m = cfg["development_reference_metrics"]
    print(f"{cfg['architecture_id']:<10}: dev100 MAE {m['MAE']:.2f} | reweighted {m['MAE_reweighted']:.2f} "
          f"| RMSE {m['RMSE']:.2f} | NAE {m['NAE']:.2f}")
print("\nSaved to:", FREEZE_DIR)

GPV FROZEN
GPV_R4_v1 : dev100 MAE 69.91 | reweighted 42.30 | RMSE 263.05 | NAE 0.42
GPV_K_v1  : dev100 MAE 54.32 | reweighted 37.06 | RMSE 151.10 | NAE 0.72

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3GK_GPV_Multiplicity_and_Prompts/frozen


# B3GK conclusion

**Merged-mask rule (Step 3):** rule A (area > 4 × median + ≥ 2 D1 peaks) — units recall 0.927, precision 0.34; peak-only rules flag ~1,000 single objects.

**Multiplicity (Steps 4a–4b):**
- **Depth edges (K3) are the strongest dense-scene multiplicity cue** — S6 373 vs shape watershed K2 578; flocks recovered (935.jpg 12 → 1,411 of 2,092).
- Semantic peaks work at medium density (K1b: best S5) but cannot reach extreme density even at 768 px (K1: 891 units vs 4,584 GT in S6 regions).
- OmniCount's depth carving (K4) is worse than no multiplicity in S6 (its opening removes small objects); area ratio (K5) fails (regions include background).
- **Every estimator amplifies false flags in sparse scenes** — the bottleneck is *where* to apply multiplicity.

**Gating (Steps 4c–4d):** SG's gate is mismatched to GPV (misses 935.jpg; fires in 8 sparse images). The pre-declared, one-shot **Gate 1** (merged peaks > individual masks) works: semantics decides *where*, geometry decides *how many*.

**Prompting (Step 5):** P1 recovers real objects in S6 but adds false positives in sparse backgrounds; neutral on the stratum-reweighted estimate — not adopted.

**Frozen for B4:** **GPV_R4_v1** (best sparse-scene accuracy, lowest NAE) and **GPV_K_v1** (best dev100 MAE/RMSE; gain concentrated in a few S6 images).

**Next (roadmap step 6):** SG with K3 (and K2 as the shape control) replacing G2 under SG_v1.1's frozen gate; then hybrid / cross-anchored designs (step 7).